# III-6 実践課題（4題）：異常検知・U-Net・NMF の GPU 実装・弦の差分法（ノートブック版）

教科書の III-6 の 6 つのプログラムを、上から順に実行すれば動く 1 冊にしたものです。**Google Colab の GPU（CUDA）で動かすこと**を主に考えています。Mac の Jupyter（MPS、または CPU）でも同じように動きます。

| 節 | 課題 | 教科書の手順 | 元のファイル |
|---|---|---|---|
| 1 | 課題1：正常の音だけで学ぶ異常検知（ROC-AUC） | 2・4 | `anomaly_ae.py` |
| 2 | 課題2：U-Net による粒子の領域分割（Dice） | 2・4 | `unet_seg.py` |
| 3 | 課題3：NMF を PyTorch に移し、NumPy 版と照合して速さを測る | 3・4・5 | `nmf_torch.py` |
| 4 | 課題4：弦の振動の差分法（解析解との照合、CFL 条件、速さ） | 3・4・5 | `string_fd.py` |
| 5 | 表にまとめ、損益分岐点と費用を読む | 4・5・8 | `p3c6_table.py` |

- 次の「準備」のセルたちが、III-2 の `train_toy.py`、III-4 の `runlog.py`、III-5 の `prep_cache.py`・`train_fast.py` と、この章の 6 ファイルを、このノートブックの隣に `%%writefile` で書き出します。**中身は教科書の各章の 6 節と 1 文字も変えていません**（ほかのファイルは要りません）。
- 各課題は、教科書のターミナルの `uv run --locked python anomaly_ae.py --quick` の代わりに、`run("anomaly_ae.py --quick")` で同じ引数を渡して別のプロセスで実行します（`run()` はこのノートブックで足した小さな関数です）。出力はそのまま表示され、結果の JSON を読んで `ok()` で確かめます。
- **既定は小さい設定**です（`--quick`、種 3 個で小さなデータ、大きさの段数を減らした測定）。教科書の本番の設定は、各セルのコメントに書いてあります。小さい設定では時間・精度の値は III-6-7 の表と合いません。`ok()` は、設定（種の数・データの数・エポック数・大きさの段数）に依らず成り立つことだけを確かめます。
- 執筆時の Mac（Apple シリコン、MPS）で、全部で 6〜8 分ほどです。Colab の GPU では、データの合成（CPU）の速さで時間が決まることが多く、Mac と同じくらいか、もっとかかることがあります。
- **入れていないもの**：手順 6・7 の Vast.ai（`vast_sync.sh`・`setup_uv_instance.sh`・`gpu_log.sh`。このノートブックを Colab で動かすことが、その代わりの「クラウドで本番」になります）、`uv.lock` と Git へのコミット（手順 1）、公開データ（MIMII・EPFL の EM データ）を使う場合（`--data npy`）。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

## 0. 準備

### Colab で GPU を選ぶ

1. メニューの「ランタイム」→「ランタイムのタイプを変更」を開き、ハードウェア アクセラレータで GPU（例：T4）を選んで保存します（項目の名前と、無料で選べる GPU の種類は公式ドキュメントで要確認）。
2. ランタイムを変えると、それまでに書き出したファイルと変数は消えるので、このノートブックは**最初のセルから**実行し直してください。
3. GPU を選ばなくても全部のセルは動きます（計算の場所が `cpu` になり、学習が数倍遅くなります）。

計算の場所（device）は、III-2 の `pick_device()` が **cuda → mps → cpu** の順に自動で選びます。Colab の GPU では `cuda`、Mac（Apple シリコン）では `mps` です。

### MACHINE と label、結果のフォルダ

III-6 では、結果の JSON の `env.machine` にマシンの名前（環境変数 `MACHINE`）を入れ、`p3c6_table.py` の machine の列で「どのマシンで測ったか」を分けます（教科書では Mac が `mac`、Vast.ai の 1 台目が `vast_m1`）。このノートブックでは次のように自動で決めます（下の準備のセル。自分で書き換えてもかまいません）。

| どこで動かすか | `MACHINE` | label（ファイル名に入る） | 結果のフォルダ `R` |
|---|---|---|---|
| Colab の T4 | `colab_t4` | `cuda_t4` | `runs/p3c6_colab_t4` |
| Colab の L4・A100 など | `colab_l4`・`colab_a100` | `cuda_l4`・`cuda_a100` | `runs/p3c6_colab_l4` など |
| Colab の CPU だけ | `colab_cpu` | `cpu` | `runs/p3c6_colab_cpu` |
| Mac（MPS） | `mac` | `mac_mps` | `runs/p3c6`（教科書と同じ） |

同じ Colab でも、別の日に別の GPU が割り当てられることがあります。教科書の「マシンを反復の単位にする」（III-3）にそろえるなら、2 回目は `MACHINE` を `colab_t4_2` のように変えて、1 回目の結果を上書きしないようにします。

### 時間と費用の測り方（III-6-3-2）

- 学習の課題は、種を変えた $m$ 回の学習時間の平均と 95% 区間 $\bar t \pm t_{0.975,m-1}\,s/\sqrt m$ を出します（種が反復の単位。1 ステップの時間は中央値で、区間には使いません）。GPU の計算は非同期なので、プログラムは `sync()`（III-5）で GPU を待ってから時刻を読み、最初の 10 ステップやウォームアップを除いて測っています。
- 費用は、学習の時間だけでなく準備と後片付けを含めて $(t_{準備} + m\bar t + t_{片付け})/3600 \times p \times r$ 円です（$p$：USD/時、$r$：円/USD）。下の `PRICE`（USD/時）は既定で 0 です。Colab の無料の枠なら 0 のまま（ただし使える時間に制限があります）、有料のプラン（コンピューティング ユニットを消費する方式）なら「1 時間に消費するユニットの数 × 1 ユニットの値段（USD）」を入れてください（どちらも GPU の種類で違います。公式サイトで要確認（執筆時点の目安））。
- このノートブックは、準備のセルを実行した時刻から、最後の節までの経過時間も測ります（5 節の費用の計算で「準備＋待ち時間」として使います）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には torch・numpy・PyYAML が最初から入っている。版は公式で要確認）
import importlib, subprocess, sys, time
T_NB_START = time.time()                      # ノートブック版：準備からの経過時間を測る（5 節の費用）
for mod, pkg in [("torch", "torch"), ("numpy", "numpy"), ("yaml", "pyyaml")]:   # yaml は III-4 の runlog.py が import する
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import numpy, torch, platform
print("Python", platform.python_version(), " numpy", numpy.__version__, " torch", torch.__version__)
print("CUDA:", torch.cuda.is_available(), " MPS:", torch.backends.mps.is_available())
print("（教科書は torch 2.14.1 で確かめた。Colab の torch の版は違うことがある）")

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

次の 4 つのセルで、ほかの章のファイル（III-2 の `train_toy.py`、III-4 の `runlog.py`、III-5 の `prep_cache.py`・`train_fast.py`）を書き出します。III-6 のファイルが import するのは、`SynthMotor`・`set_seed`・`pick_device`・`device_name`（`train_toy.py`）、`git_info`（`runlog.py`）、`log_stft`（`prep_cache.py`）、`sync`（`train_fast.py`）です。

**`runlog.py` の `git_info()` と Colab**：`p3c6_util.py` の `env_summary()` は、結果の JSON の `env.git` に III-4 の `git_info()` のコミット ID を入れます。`git_info()` は Git のコマンドが無いときや、Git のリポジトリの外で実行したときはエラーにせず `{"commit": "unknown", "dirty": None}` を返すので、Colab の `/content` でもこのまま動きます（下の準備の確かめで確かめます）。ただし、このノートブックで書き出したコードは Git に入っていないので、「どの版のコードで測ったか」は記録されません。記録したいときは、III-2 のセル3 のように自分の private リポジトリを clone して、そのフォルダで実行してください（このノートブックは準備のセルで `GIT_CEILING_DIRECTORIES` を設定し、ノートブックのフォルダより上の Git のリポジトリを見ないようにしています。そのため Mac でも、ノートブックのフォルダそのものが clone したリポジトリでなければ `unknown` になり、教科書のフォルダなどの関係ないコミット ID は入りません）。

In [ ]:
%%writefile train_toy.py
#!/usr/bin/env python3
"""III-2：どこでも動く練習用の学習スクリプト
合成した「モータ音」の正常／異常を、1次元CNNで分類する。
- device を自動で選ぶ（cuda → mps → cpu）。同じコードが Mac と Colab で動く
- 一定時間ごと＋エポックの終わりにチェックポイントを保存。切断されても途中から正確に再開できる
- CUDA では混合精度（AMP）を使う
- --max-hours を過ぎたら、保存して自分で終了する（課金の止め忘れ対策）
使い方:
  uv run python train_toy.py --quick --ckpt-dir runs/mac_quick              # Mac で動作確認
  python train_toy.py --ckpt-dir /content/drive/MyDrive/home-lab/runs/toy01    # Colab で本番
"""
import argparse
import contextlib
import csv
import json
import math
import os
import platform
import random
import signal
import subprocess
import sys
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset

FS = 16000
stop_requested = False


def on_signal(signum, frame):
    global stop_requested
    stop_requested = True
    print(f"\nシグナル {signum} を受信：区切りのよいところで保存して終了します", flush=True)


# ---------------------------------------------------------------- 環境
def pick_device(name="auto"):
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def device_name(dev):
    if dev.type == "cuda":
        return torch.cuda.get_device_name(0)
    if dev.type == "mps":
        return "Apple MPS"
    return platform.processor() or platform.machine()


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def git_commit():
    try:
        r = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                           capture_output=True, text=True, timeout=5)
        return r.stdout.strip() or "unknown"
    except Exception:
        return "unknown"


def amp_ctx(enabled):
    return torch.autocast("cuda", dtype=torch.float16) if enabled else contextlib.nullcontext()


# ---------------------------------------------------------------- データ（合成）
def make_signal(rng, length, anomalous):
    """基本波 80〜120 Hz と高調波＋雑音。異常は「周期的なクリック」か「高い周波数のうなり」"""
    t = np.arange(length) / FS
    f0 = rng.uniform(80, 120)
    x = np.zeros(length)
    for k in range(1, 6):
        x += np.sin(2 * np.pi * k * f0 * t + rng.uniform(0, 2 * np.pi)) / k
    x += rng.normal(0, 0.3, length)
    if anomalous:
        if rng.integers(2) == 0:
            period = int(FS / rng.uniform(15, 40))
            win = np.hanning(16) * rng.uniform(2, 4)
            for p in range(int(rng.integers(period)), length, period):
                seg = x[p:p + 16]
                seg += win[:len(seg)]
        else:
            fm = rng.uniform(1500, 3000)
            env = 1 + 0.5 * np.sin(2 * np.pi * rng.uniform(5, 20) * t)
            x += 0.6 * np.sin(2 * np.pi * fm * t) * env
    return (x / x.std()).astype(np.float32)


class SynthMotor(Dataset):
    """番号 i から決まる乱数で毎回同じ信号を作る（ファイルに保存しないのでディスクを使わない）"""
    def __init__(self, n, length, seed):
        self.n, self.length, self.seed = n, length, seed

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        rng = np.random.default_rng([self.seed, i])
        y = i % 2                                 # 正常と異常を半分ずつ
        x = make_signal(rng, self.length, bool(y))
        return torch.from_numpy(x).unsqueeze(0), y


# ---------------------------------------------------------------- モデル
class AudioCNN(nn.Module):
    def __init__(self, w=32, ncls=2):
        super().__init__()
        layers, cin = [], 1
        for cout in (w, w, 2 * w, 2 * w, 4 * w, 4 * w):
            layers += [nn.Conv1d(cin, cout, 9, stride=2, padding=4),
                       nn.BatchNorm1d(cout), nn.ReLU(inplace=True)]
            cin = cout
        self.features = nn.Sequential(*layers)
        self.head = nn.Linear(cin, ncls)

    def forward(self, x):
        return self.head(self.features(x).mean(dim=-1))


# ---------------------------------------------------------------- チェックポイント
def save_ckpt(path, state):
    tmp = path + ".tmp"
    torch.save(state, tmp)
    os.replace(tmp, path)                         # 書き込み途中で切れても壊れない


@torch.no_grad()
def evaluate(model, loader, dev, use_amp):
    model.eval()
    correct = n = 0
    for x, y in loader:
        x, y = x.to(dev), y.to(dev)
        with amp_ctx(use_amp):
            pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        n += len(y)
    model.train()
    return correct / n


def parse():
    ap = argparse.ArgumentParser()
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--n-train", type=int, default=20000)
    ap.add_argument("--n-val", type=int, default=2000)
    ap.add_argument("--length", type=int, default=4096)
    ap.add_argument("--batch", type=int, default=128)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--no-amp", action="store_true")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--ckpt-dir", default="runs/toy")
    ap.add_argument("--ckpt-every-min", type=float, default=5.0)
    ap.add_argument("--max-hours", type=float, default=3.0, help="この時間を過ぎたら保存して終了")
    ap.add_argument("--quick", action="store_true", help="小さい設定で動作確認")
    a = ap.parse_args()
    if a.quick:
        a.n_train, a.n_val, a.epochs = 4000, 400, 2
    return a


def main():
    a = parse()
    signal.signal(signal.SIGTERM, on_signal)
    signal.signal(signal.SIGINT, on_signal)
    set_seed(a.seed)
    dev = pick_device(a.device)
    use_amp = dev.type == "cuda" and not a.no_amp
    if dev.type == "cuda":
        torch.backends.cudnn.benchmark = True
    os.makedirs(a.ckpt_dir, exist_ok=True)
    last = os.path.join(a.ckpt_dir, "last.pt")
    metrics_path = os.path.join(a.ckpt_dir, "metrics.csv")
    t_start = time.time()

    train_ds = SynthMotor(a.n_train, a.length, seed=a.seed)
    val_ds = SynthMotor(a.n_val, a.length, seed=a.seed + 10_000)
    pin = dev.type == "cuda"
    val_loader = DataLoader(val_ds, batch_size=a.batch * 2, num_workers=a.workers, pin_memory=pin)

    model = AudioCNN().to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    start_epoch, start_step = 0, 0
    run = {"loss_sum": 0.0, "n": 0, "t": 0.0}
    resumed = False
    if os.path.exists(last):
        ck = torch.load(last, map_location="cpu", weights_only=False)   # 自分で作ったファイルなので可
        if ck.get("done"):
            print(f"この実行はすでに完了しています: {last}")
            return
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        if use_amp and ck.get("scaler"):
            scaler.load_state_dict(ck["scaler"])
        start_epoch, start_step, run = ck["epoch"], ck["step"], ck["run"]
        resumed = True
        print(f"チェックポイントから再開: epoch {start_epoch} step {start_step}", flush=True)

    # 実行の記録（再開のたびに追記）
    info_path = os.path.join(a.ckpt_dir, "run_info.json")
    info = json.load(open(info_path, encoding="utf-8")) if os.path.exists(info_path) else {"sessions": []}
    info["sessions"].append({
        "started": time.strftime("%Y-%m-%dT%H:%M:%S%z"), "git_commit": git_commit(),
        "torch": torch.__version__, "python": sys.version.split()[0],
        "device": dev.type, "device_name": device_name(dev), "amp": use_amp,
        "args": vars(a), "resumed": resumed})
    with open(info_path, "w", encoding="utf-8") as f:
        json.dump(info, f, ensure_ascii=False, indent=2)
    print(f"device = {device_name(dev)}（{dev.type}）, AMP = {use_amp}, commit = {git_commit()}", flush=True)

    def state(epoch, step, run_state, done=False):
        return {"model": model.state_dict(), "opt": opt.state_dict(),
                "scaler": scaler.state_dict(), "epoch": epoch, "step": step,
                "run": run_state, "args": vars(a), "done": done,
                "saved_at": time.strftime("%Y-%m-%dT%H:%M:%S%z")}

    last_save = time.time()
    for epoch in range(start_epoch, a.epochs):
        first = epoch == start_epoch
        if not first:
            run = {"loss_sum": 0.0, "n": 0, "t": 0.0}
        step = start_step if first else 0
        # エポックごとの並び順を「種＋エポック番号」で決める → 途中からでも同じ順番で再開できる
        g = torch.Generator().manual_seed(a.seed * 1000 + epoch)
        order = torch.randperm(len(train_ds), generator=g).tolist()
        loader = DataLoader(Subset(train_ds, order[step * a.batch:]), batch_size=a.batch,
                            shuffle=False, num_workers=a.workers, pin_memory=pin)
        model.train()
        t_seg = time.time()
        for x, y in loader:
            x = x.to(dev, non_blocking=True)
            y = y.to(dev, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with amp_ctx(use_amp):
                loss = F.cross_entropy(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            run["loss_sum"] += loss.item() * len(y)
            run["n"] += len(y)
            step += 1
            over_time = time.time() - t_start > a.max_hours * 3600
            if stop_requested or over_time or time.time() - last_save > a.ckpt_every_min * 60:
                run["t"] += time.time() - t_seg
                t_seg = time.time()
                save_ckpt(last, state(epoch, step, run))
                last_save = time.time()
                print(f"  チェックポイント保存: epoch {epoch} step {step}", flush=True)
                if stop_requested or over_time:
                    why = "シグナル" if stop_requested else f"--max-hours {a.max_hours}"
                    print(f"{why} により終了します。同じコマンドで再開できます。", flush=True)
                    return
        run["t"] += time.time() - t_seg
        val_acc = evaluate(model, val_loader, dev, use_amp)
        row = {"epoch": epoch, "train_loss": run["loss_sum"] / max(run["n"], 1),
               "val_acc": val_acc, "epoch_time_s": run["t"],
               "samples_per_s": run["n"] / max(run["t"], 1e-9),
               "device": dev.type, "device_name": device_name(dev),
               "resumed": resumed and first, "time": time.strftime("%Y-%m-%dT%H:%M:%S%z")}
        new = not os.path.exists(metrics_path)
        with open(metrics_path, "a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(row.keys()))
            if new:
                w.writeheader()
            w.writerow(row)
        print(f"epoch {epoch}: loss {row['train_loss']:.4f}  val_acc {val_acc:.4f}  "
              f"{row['samples_per_s']:.0f} サンプル/秒", flush=True)
        save_ckpt(last, state(epoch + 1, 0, {"loss_sum": 0.0, "n": 0, "t": 0.0},
                              done=(epoch + 1 == a.epochs)))
        last_save = time.time()
    print("完了しました。", flush=True)


if __name__ == "__main__":     # DataLoader の並列読み込み（macOS の spawn）に必要
    main()

In [ ]:
%%writefile runlog.py
#!/usr/bin/env python3
"""III-4：実行の記録（コミットID・設定・環境・データ）を run_record.json に残す。2つの記録の違いを表示する
- 設定は YAML。--set a.b=値 で一部だけ上書きできる（Hydra の上書きの書き方をまねた最小限のもの）
- 記録しないもの：環境変数の値（決まった名前だけ）、ホスト名（個人名が入ることがある）、秘密の値
使い方:
  （train_repro.py などから）from runlog import load_config, start_run, finish_run
  uv run python runlog.py show runs/seed_mps/seed000            # 1つの記録を見る
  uv run python runlog.py diff runs/seed_mps/seed000 runs/seed_cuda/seed000   # 何が違うか
"""
import argparse
import copy
import datetime
import hashlib
import json
import os
import platform
import subprocess
import sys

import yaml

# 記録してよい環境変数（値に秘密が入らないものだけ）。HF_TOKEN や WANDB_API_KEY は「名前があるか」だけ記録する
ENV_KEEP = ["CUBLAS_WORKSPACE_CONFIG", "PYTHONHASHSEED", "CUDA_VISIBLE_DEVICES", "OMP_NUM_THREADS",
            "UV_TORCH_BACKEND", "WANDB_MODE", "HF_HOME"]
ENV_SECRET_NAMES = ["HF_TOKEN", "WANDB_API_KEY", "VAST_API_KEY", "GH_TOKEN"]
CODE_SUFFIX = (".py", ".sh", ".yaml", ".yml", ".toml")   # コミットしていなければ「変更あり」とみなすファイル


# ---------------------------------------------------------------- 設定ファイル
def _parse_value(text):
    """'0.01' → 0.01、'true' → True、'[1,2]' → [1, 2]（YAML の書き方で読む）"""
    return yaml.safe_load(text)


def apply_overrides(cfg, overrides):
    """['train.epochs=5', 'model.width=32'] のような上書きを適用する。無い鍵はエラー（打ち間違いを防ぐ）"""
    cfg = copy.deepcopy(cfg)
    for item in overrides or []:
        if "=" not in item:
            raise SystemExit(f"--set の書き方が違います（鍵=値）: {item}")
        key, val = item.split("=", 1)
        node = cfg
        parts = key.split(".")
        for p in parts[:-1]:
            if not isinstance(node, dict) or p not in node:
                raise SystemExit(f"設定に {key} がありません（YAML の鍵を確認）")
            node = node[p]
        if not isinstance(node, dict) or parts[-1] not in node:
            raise SystemExit(f"設定に {key} がありません（YAML の鍵を確認）")
        node[parts[-1]] = _parse_value(val)
    return cfg


def load_config(path, overrides=None):
    with open(path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    return apply_overrides(cfg, overrides)


def config_hash(cfg):
    """設定の中身のハッシュ（鍵の順番や空白に左右されない）。同じ値なら同じ設定"""
    text = json.dumps(cfg, sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(text.encode()).hexdigest()[:12]


# ---------------------------------------------------------------- Git・ファイル・環境
def _git(*args):
    try:
        r = subprocess.run(["git", *args], capture_output=True, text=True, timeout=10)
        return r.stdout.strip() if r.returncode == 0 else None
    except (OSError, subprocess.SubprocessError):
        return None


def git_info():
    commit = _git("rev-parse", "HEAD")
    if commit is None:
        return {"commit": "unknown", "dirty": None}
    lines = (_git("-c", "core.quotepath=off", "status", "--porcelain", "--untracked-files=all") or "").splitlines()
    changed = [x for x in lines if not x.startswith("??")]
    # Git に入っていない新しいコード（import されうる）も「変更あり」とみなす。data/ runs/ は .gitignore で見えない
    untracked_code = [x[3:].strip('"') for x in lines if x.startswith("??") and x.strip('"').endswith(CODE_SUFFIX)]
    return {"commit": commit, "short": commit[:7],
            "branch": _git("rev-parse", "--abbrev-ref", "HEAD"),
            "dirty": bool(changed or untracked_code), "changed_files": changed,
            "untracked_code": untracked_code}


def file_sha256(path, n=12):
    if not os.path.exists(path):
        return None
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()[:n]


def env_info(device=None):
    info = {"python": sys.version.split()[0], "os": platform.system(), "os_release": platform.release(),
            "machine_arch": platform.machine(), "uv_lock_sha256": file_sha256("uv.lock")}
    try:
        import numpy
        info["numpy"] = numpy.__version__
    except ImportError:
        pass
    try:
        import torch
        info["torch"] = torch.__version__
        info["torch_cuda"] = torch.version.cuda
        info["cudnn"] = torch.backends.cudnn.version() if torch.backends.cudnn.is_available() else None
        info["mps_available"] = torch.backends.mps.is_available()
        if device is not None and str(device).startswith("cuda"):
            info["gpu"] = torch.cuda.get_device_name(0)
            info["gpu_capability"] = ".".join(map(str, torch.cuda.get_device_capability(0)))
    except ImportError:
        info["torch"] = None
    info["env"] = {k: os.environ[k] for k in ENV_KEEP if k in os.environ}
    info["secrets_present"] = [k for k in ENV_SECRET_NAMES if os.environ.get(k)]   # 名前だけ。値は残さない
    return info


# ---------------------------------------------------------------- 記録
def start_run(outdir, cfg, seed, machine, device=None, extra=None, allow_dirty=False):
    """実行の最初に呼ぶ。run_record.json を書き、コミットしていない変更があれば差分も残す"""
    os.makedirs(outdir, exist_ok=True)
    g = git_info()
    if g.get("dirty"):
        if not allow_dirty:
            raise SystemExit("コミットしていない変更があります。コミットしてから実行するか、"
                             "練習なら --allow-dirty を付けてください（差分を uncommitted.diff に残します）")
        with open(os.path.join(outdir, "uncommitted.diff"), "w", encoding="utf-8") as f:
            f.write(_git("diff", "HEAD") or "")
            for p in g.get("untracked_code", []):          # 新しいファイルは中身ごと残す
                f.write(f"\n# ===== Git に入っていないファイル: {p} =====\n")
                try:
                    with open(p, encoding="utf-8", errors="replace") as src:
                        f.write(src.read())
                except OSError:
                    f.write("（読めませんでした）\n")
    rec = {"started": datetime.datetime.now().astimezone().isoformat(timespec="seconds"),
           "argv": sys.argv, "machine": machine, "seed": seed,
           "git": g, "config": cfg, "config_hash": config_hash(cfg),
           "env": env_info(device), "extra": extra or {}}
    write_json(os.path.join(outdir, "run_record.json"), rec)
    return rec


def finish_run(outdir, rec, results):
    rec = dict(rec)
    rec["finished"] = datetime.datetime.now().astimezone().isoformat(timespec="seconds")
    rec["results"] = results
    write_json(os.path.join(outdir, "run_record.json"), rec)
    return rec


def write_json(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)
    os.replace(tmp, path)


# ---------------------------------------------------------------- 表示・比較
def flatten(d, prefix=""):
    out = {}
    for k, v in d.items():
        key = f"{prefix}{k}"
        if isinstance(v, dict):
            out.update(flatten(v, key + "."))
        else:
            out[key] = v
    return out


def load_record(path):
    if os.path.isdir(path):
        path = os.path.join(path, "run_record.json")
    with open(path, encoding="utf-8") as f:
        return json.load(f)


IGNORE = ("started", "finished", "argv", "results.wall_s", "git.changed_files", "git.untracked_code")


def diff_records(a, b):
    fa, fb = flatten(a), flatten(b)
    rows = []
    for k in sorted(set(fa) | set(fb)):
        if k.startswith(IGNORE):
            continue
        if fa.get(k) != fb.get(k):
            rows.append((k, fa.get(k), fb.get(k)))
    return rows


def main():
    ap = argparse.ArgumentParser()
    sub = ap.add_subparsers(dest="cmd", required=True)
    s = sub.add_parser("show")
    s.add_argument("run")
    d = sub.add_parser("diff")
    d.add_argument("a")
    d.add_argument("b")
    a = ap.parse_args()
    if a.cmd == "show":
        r = load_record(a.run)
        for k, v in flatten(r).items():
            print(f"{k:32s} {v}")
    else:
        rows = diff_records(load_record(a.a), load_record(a.b))
        if not rows:
            print("違いはありません（時刻・コマンドラインを除く）")
        for k, va, vb in rows:
            print(f"{k:32s} {str(va):28s} | {vb}")
        keys = [k for k, _, _ in rows]
        important = [k for k in keys if k.startswith(("git.commit", "env.uv_lock", "config_hash", "env.torch",
                                                      "extra.data_id"))]
        if important:
            print("\n注意：コード・環境・設定・データのどれかが違います。結果の差がこれのせいか、"
                  "種や device のせいかを区別できません:", ", ".join(important))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile prep_cache.py
#!/usr/bin/env python3
"""III-5：train_toy.py の合成データを、前もって計算してファイルに保存する（前処理の事前計算）
- 波形を data/<名前>/x.npy（float32、形 (N, L)）、ラベルを y.npy に保存する
- --stft を付けると、対数振幅スペクトログラム（float16、形 (N, F, T)）も stft.npy に保存する
- 学習のときは CachedMotor で np.load(mmap_mode="r") して読む（メモリに全部は載せない）
使い方:
  uv run python prep_cache.py --n 20000 --length 4096 --seed 0 --name toy_train
  uv run python prep_cache.py --n 20000 --length 4096 --seed 0 --name toy_train --stft
"""
import argparse
import json
import os
import time

import numpy as np
import torch
from torch.utils.data import Dataset

from train_toy import SynthMotor

N_FFT, HOP = 256, 64


def log_stft(x, n_fft=N_FFT, hop=HOP):
    """x：(B, L) の波形 → (B, n_fft/2+1, フレーム数) の log(1+|X|)。CPU でも GPU でも同じ関数"""
    win = torch.hann_window(n_fft, device=x.device)
    X = torch.stft(x, n_fft=n_fft, hop_length=hop, window=win, center=True, return_complex=True)
    return torch.log1p(X.abs())


class CachedMotor(Dataset):
    """prep_cache.py が作ったファイルを読む。SynthMotor と同じく (x, y) を返す
    kind="wave" なら (1, L) の波形、kind="stft" なら (F, T) のスペクトログラム"""
    def __init__(self, folder, kind="wave"):
        self.folder, self.kind = folder, kind
        self.y = np.load(os.path.join(folder, "y.npy"))
        self.x = None                     # 開くのは各ワーカーの中（fork・spawn どちらでも安全）

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        if self.x is None:
            name = "x.npy" if self.kind == "wave" else "stft.npy"
            self.x = np.load(os.path.join(self.folder, name), mmap_mode="r")
        a = torch.from_numpy(np.array(self.x[i], dtype=np.float32))   # コピーしてから渡す
        if self.kind == "wave":
            a = a.unsqueeze(0)
        return a, int(self.y[i])


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n", type=int, default=20000)
    ap.add_argument("--length", type=int, default=4096)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--name", default="toy_train")
    ap.add_argument("--root", default="data")
    ap.add_argument("--stft", action="store_true", help="スペクトログラムも保存する")
    ap.add_argument("--chunk", type=int, default=512)
    a = ap.parse_args()

    out = os.path.join(a.root, a.name)
    os.makedirs(out, exist_ok=True)
    ds = SynthMotor(a.n, a.length, seed=a.seed)
    t0 = time.perf_counter()
    x = np.lib.format.open_memmap(os.path.join(out, "x.npy"), mode="w+",
                                  dtype=np.float32, shape=(a.n, a.length))
    y = np.zeros(a.n, dtype=np.int64)
    for i in range(a.n):
        xi, yi = ds[i]
        x[i] = xi[0].numpy()
        y[i] = yi
    x.flush()
    np.save(os.path.join(out, "y.npy"), y)
    t_wave = time.perf_counter() - t0
    print(f"波形 {a.n} 個を保存：{t_wave:.1f} 秒（{a.n / t_wave:.0f} 個/秒）、"
          f"{x.nbytes / 1e6:.0f} MB → {out}/x.npy", flush=True)

    info = {"n": a.n, "length": a.length, "seed": a.seed, "source": "train_toy.SynthMotor",
            "wave_s": round(t_wave, 2)}
    if a.stft:
        t0 = time.perf_counter()
        F_ = log_stft(torch.zeros(1, a.length)).shape[1:]
        s = np.lib.format.open_memmap(os.path.join(out, "stft.npy"), mode="w+",
                                      dtype=np.float16, shape=(a.n, *F_))
        for k in range(0, a.n, a.chunk):
            xb = torch.from_numpy(np.asarray(x[k:k + a.chunk]))
            s[k:k + len(xb)] = log_stft(xb).numpy().astype(np.float16)
        s.flush()
        t_stft = time.perf_counter() - t0
        info.update({"stft_shape": list(F_), "n_fft": N_FFT, "hop": HOP, "stft_s": round(t_stft, 2)})
        print(f"STFT（CPU）：{t_stft:.1f} 秒、形 {tuple(F_)}、{s.nbytes / 1e6:.0f} MB → {out}/stft.npy")
    with open(os.path.join(out, "cache_info.json"), "w", encoding="utf-8") as f:
        json.dump(info, f, ensure_ascii=False, indent=2)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile train_fast.py
#!/usr/bin/env python3
"""III-5：GPU がデータを待っている時間を測り、速くする工夫を1つずつ試す
train_toy.py と同じデータ（SynthMotor）・同じモデル（AudioCNN）で、決めたステップ数だけ学習し、
1ステップを「データ待ち」と「GPU の計算」に分けて記録する。
- --data synth：train_toy.py と同じく、信号を毎回 CPU で作る
- --data cache：prep_cache.py で保存したファイル（memmap）を読む
- --amp off|fp16|bf16、--accum K（勾配累積）、--compile（torch.compile）
使い方:
  uv run python train_fast.py --data synth --workers 0 --label synth_w0
  uv run python train_fast.py --data cache --cache-dir data/toy_train --workers 2 --label cache_w2
"""
import argparse
import contextlib
import csv
import json
import os
import time

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from prep_cache import CachedMotor
from train_toy import AudioCNN, SynthMotor, device_name, pick_device, set_seed


def sync(dev):
    """GPU の計算は非同期なので、時刻を測る前に終わるのを待つ"""
    if dev.type == "cuda":
        torch.cuda.synchronize()
    elif dev.type == "mps":
        torch.mps.synchronize()


def autocast_ctx(dev, amp):
    if amp == "off":
        return contextlib.nullcontext()
    dtype = torch.float16 if amp == "fp16" else torch.bfloat16
    return torch.autocast(dev.type, dtype=dtype)


def peak_mem_mb(dev):
    if dev.type == "cuda":
        return torch.cuda.max_memory_allocated() / 2**20
    if dev.type == "mps":
        return torch.mps.driver_allocated_memory() / 2**20   # MPS は最大値ではなく今の確保量
    return float("nan")


def amdahl(f, s):
    """速くなる部分の割合 f を s 倍にしたときの全体の倍率"""
    return 1.0 / ((1.0 - f) + f / s)


def parse():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", choices=["synth", "cache"], default="synth")
    ap.add_argument("--cache-dir", default="data/toy_train")
    ap.add_argument("--n-train", type=int, default=20000)
    ap.add_argument("--length", type=int, default=4096)
    ap.add_argument("--batch", type=int, default=128)
    ap.add_argument("--accum", type=int, default=1, help="この回数ぶん勾配をためてから更新する")
    ap.add_argument("--workers", type=int, default=0)
    ap.add_argument("--prefetch", type=int, default=2, help="ワーカー1つが先に用意するバッチ数")
    ap.add_argument("--amp", choices=["off", "fp16", "bf16"], default="off")
    ap.add_argument("--compile", action="store_true")
    ap.add_argument("--steps", type=int, default=120, help="記録するステップ数（ウォームアップは別）")
    ap.add_argument("--warmup", type=int, default=10)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--label", default="run")
    ap.add_argument("--outdir", default="runs/fast")
    return ap.parse_args()


def main():
    a = parse()
    set_seed(a.seed)
    dev = pick_device(a.device)
    if a.amp == "fp16" and dev.type == "cpu":
        raise SystemExit("CPU では --amp bf16 を使ってください（fp16 の autocast は遅いか使えない）")
    if dev.type == "cuda":
        torch.backends.cudnn.benchmark = True

    if a.data == "synth":
        ds = SynthMotor(a.n_train, a.length, seed=a.seed)
    else:
        ds = CachedMotor(a.cache_dir, kind="wave")
    pin = dev.type == "cuda"
    kw = dict(batch_size=a.batch, shuffle=True, num_workers=a.workers, pin_memory=pin,
              drop_last=True, generator=torch.Generator().manual_seed(a.seed))
    if a.workers > 0:
        kw.update(persistent_workers=True, prefetch_factor=a.prefetch)
    loader = DataLoader(ds, **kw)

    model = AudioCNN().to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr)
    use_scaler = dev.type == "cuda" and a.amp == "fp16"
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)
    step_model = torch.compile(model) if a.compile else model

    total = a.warmup + a.steps
    rows, it, n_micro = [], None, total * a.accum
    if dev.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t_begin = time.perf_counter()
    sync(dev)
    t_prev = time.perf_counter()
    for k in range(total):
        wait = comp = 0.0
        opt.zero_grad(set_to_none=True)
        for m in range(a.accum):
            if it is None:
                it = iter(loader)
            try:
                x, y = next(it)
            except StopIteration:              # エポックの終わり：もう一周
                it = iter(loader)
                x, y = next(it)
            x = x.to(dev, non_blocking=True)
            y = y.to(dev, non_blocking=True)
            sync(dev)
            t_data = time.perf_counter()
            wait += t_data - t_prev
            with autocast_ctx(dev, a.amp):
                loss = F.cross_entropy(step_model(x), y) / a.accum
            scaler.scale(loss).backward()
            if m == a.accum - 1:
                scaler.step(opt)
                scaler.update()
            sync(dev)
            t_prev = time.perf_counter()
            comp += t_prev - t_data
        rows.append({"step": k, "warmup": k < a.warmup, "wait_s": wait, "compute_s": comp,
                     "loss": loss.item() * a.accum})
    wall = time.perf_counter() - t_begin

    rec = [r for r in rows if not r["warmup"]]
    w = np.array([r["wait_s"] for r in rec])
    c = np.array([r["compute_s"] for r in rec])
    step_s = w + c
    f_data = w.sum() / step_s.sum()
    samples = a.batch * a.accum
    summ = {
        "label": a.label, "device": dev.type, "device_name": device_name(dev),
        "torch": torch.__version__, "data": a.data, "workers": a.workers, "batch": a.batch,
        "accum": a.accum, "amp": a.amp, "compile": a.compile, "steps": a.steps,
        "samples_per_s": samples * len(rec) / step_s.sum(),
        "step_ms_median": 1000 * float(np.median(step_s)),
        "wait_ms_median": 1000 * float(np.median(w)),
        "compute_ms_median": 1000 * float(np.median(c)),
        "f_data": float(f_data),
        "first_step_s": rows[0]["wait_s"] + rows[0]["compute_s"],
        "wall_s": wall, "peak_mem_mb": peak_mem_mb(dev),
        "final_loss": float(np.mean([r["loss"] for r in rec[-20:]])),
    }
    # アムダールの法則：データ待ち以外（GPU の計算）だけが速くなる場合の上限
    f_gpu = 1 - f_data
    summ["amdahl_x2"] = amdahl(f_gpu, 2)
    summ["amdahl_limit"] = 1 / f_data if f_data > 0 else float("inf")

    os.makedirs(a.outdir, exist_ok=True)
    with open(os.path.join(a.outdir, f"{a.label}_steps.csv"), "w", newline="") as fh:
        wr = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        wr.writeheader()
        wr.writerows(rows)
    with open(os.path.join(a.outdir, f"{a.label}.json"), "w", encoding="utf-8") as fh:
        json.dump(summ, fh, ensure_ascii=False, indent=2)

    print(f"[{a.label}] {summ['device_name']}（{dev.type}） data={a.data} workers={a.workers} "
          f"batch={a.batch}×{a.accum} amp={a.amp} compile={a.compile}")
    print(f"  {summ['samples_per_s']:.0f} サンプル/秒、1ステップ 中央値 {summ['step_ms_median']:.1f} ms"
          f"（データ待ち {summ['wait_ms_median']:.1f} ms ＋ 計算 {summ['compute_ms_median']:.1f} ms）")
    print(f"  データ待ちの割合 {100 * f_data:.0f}%  → GPU を2倍速くしても全体は "
          f"{summ['amdahl_x2']:.2f} 倍、無限に速くしても {summ['amdahl_limit']:.1f} 倍まで")
    print(f"  最初のステップ {summ['first_step_s']:.2f} 秒、メモリ {summ['peak_mem_mb']:.0f} MB、"
          f"損失（最後の20ステップの平均）{summ['final_loss']:.3f}")


if __name__ == "__main__":     # DataLoader のワーカー（macOS の spawn）に必要
    main()

次の 6 つのセルで、この章のファイルを書き出します（教科書 III-6-6 と同じもの）。

In [ ]:
%%writefile p3c6_util.py
#!/usr/bin/env python3
"""III-6：4つの課題で共通に使う道具（時間の測り方・信頼区間・ROC-AUC・記録）
train_toy.py（III-2）・train_fast.py（III-5）から import するので、~/home-cloud に置く。
"""
import json
import math
import os
import platform
import time

import numpy as np
import torch

from runlog import git_info
from train_fast import sync
from train_toy import device_name, pick_device

# t 分布の 97.5% 点（自由度 1〜30）。scipy を足さないため表で持つ
T975 = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571, 6: 2.447, 7: 2.365, 8: 2.306,
        9: 2.262, 10: 2.228, 11: 2.201, 12: 2.179, 13: 2.160, 14: 2.145, 15: 2.131,
        16: 2.120, 17: 2.110, 18: 2.101, 19: 2.093, 20: 2.086, 25: 2.060, 30: 2.042}


def t975(df):
    if df in T975:
        return T975[df]
    if df > 30:
        return 1.96 + 2.4 / df          # 大きい自由度の近似（df=30 で 2.04）
    return T975[max(k for k in T975 if k <= df)]


def mean_ci(xs, lo=None, hi=None):
    """反復（マシン・種）ごとの値の平均と 95% t 区間。1個だけなら区間は nan。
    AUC・Dice のように範囲が決まっている値は lo=0, hi=1 で区間を切る（t 区間は範囲を知らないため）"""
    x = np.asarray(xs, float)
    m = float(x.mean())
    if len(x) < 2:
        return m, float("nan"), float("nan")
    h = t975(len(x) - 1) * float(x.std(ddof=1)) / math.sqrt(len(x))
    a, b = m - h, m + h
    if lo is not None:
        a = max(a, lo)
    if hi is not None:
        b = min(b, hi)
    return m, a, b


def bench(fn, dev, reps=7, warmup=2):
    """fn() を warmup 回まわして捨て、reps 回の時間（秒）を返す。毎回 GPU と同期する"""
    for _ in range(warmup):
        fn()
    sync(dev)
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        fn()
        sync(dev)
        ts.append(time.perf_counter() - t0)
    return ts


def roc_auc(score_normal, score_anom):
    """ROC-AUC＝「異常の得点が正常より大きい」確率（マン・ホイットニーの U と同じ。同点は 1/2）"""
    s = np.concatenate([score_normal, score_anom])
    order = s.argsort(kind="mergesort")
    rank = np.empty(len(s))
    rank[order] = np.arange(1, len(s) + 1)
    for v in np.unique(s):              # 同点は順位の平均
        idx = s == v
        if idx.sum() > 1:
            rank[idx] = rank[idx].mean()
    n0, n1 = len(score_normal), len(score_anom)
    return float((rank[n0:].sum() - n1 * (n1 + 1) / 2) / (n0 * n1))


def auc_boot_ci(score_normal, score_anom, n_boot=1000, seed=0):
    """正常・異常をそれぞれ復元抽出するブートストラップの 95% 区間（パーセンタイル法）"""
    rng = np.random.default_rng(seed)
    a = [roc_auc(rng.choice(score_normal, len(score_normal)), rng.choice(score_anom, len(score_anom)))
         for _ in range(n_boot)]
    return float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))


def env_summary(dev):
    return {"device": dev.type, "device_name": device_name(dev), "torch": torch.__version__,
            "numpy": np.__version__, "python": platform.python_version(),
            "machine": os.environ.get("MACHINE", "unknown"),
            "git": git_info()}            # III-4 の runlog.py：コミット ID と未コミットの変更の有無


def write_json(path, obj):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    print(f"保存：{path}")


__all__ = ["pick_device", "sync", "bench", "mean_ci", "roc_auc", "auc_boot_ci", "env_summary", "write_json"]

In [ ]:
%%writefile anomaly_ae.py
#!/usr/bin/env python3
"""III-6 課題1：音響の異常検知（正常だけで学習する畳み込みオートエンコーダ）
合成モータ音（III-2 の SynthMotor）の「正常」だけで学習し、再構成誤差を異常の得点にする。
評価は ROC-AUC（ブートストラップの 95% 区間つき）。種を変えて数回学習し、時間と費用を区間で出す。
第5章の録音を使うときは、第5章 5-6-1 の変更点（motor_rec.py と --rec-dir）を当てる。
使い方:
  uv run python anomaly_ae.py --quick                                   # Mac で動作確認（1分ほど）
  uv run python anomaly_ae.py --seeds 0 1 2 --label mac                 # Mac で本番と同じ設定
  python anomaly_ae.py --seeds 0 1 2 --label cuda --outdir /workspace/runs/p3c6 \
      --price-usd-h 0.40 --usd-jpy 150                                  # クラウド
"""
import argparse
import time

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset

from p3c6_util import auc_boot_ci, env_summary, mean_ci, pick_device, roc_auc, sync, write_json
from prep_cache import log_stft
from train_toy import SynthMotor, set_seed


class ConvAE(nn.Module):
    """入力 (B, 1, 128, 64) の対数 STFT → 8×4 まで縮めて戻す"""
    def __init__(self, w=16):
        super().__init__()
        def down(i, o):
            return nn.Sequential(nn.Conv2d(i, o, 3, stride=2, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True))
        def up(i, o):
            return nn.Sequential(nn.ConvTranspose2d(i, o, 4, stride=2, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True))
        self.enc = nn.Sequential(down(1, w), down(w, 2 * w), down(2 * w, 4 * w), down(4 * w, 4 * w))
        self.dec = nn.Sequential(up(4 * w, 4 * w), up(4 * w, 2 * w), up(2 * w, w),
                                 nn.ConvTranspose2d(w, 1, 4, stride=2, padding=1))

    def forward(self, s):
        return self.dec(self.enc(s))


def features(x, dev):
    """波形 (B, 1, L) → 対数 STFT (B, 1, 128, 64)。GPU の上で計算し、標準化は固定の値で行う"""
    s = log_stft(x.squeeze(1).to(dev, non_blocking=True))[:, :128, :64]
    return ((s - 1.0) / 0.7).unsqueeze(1)


@torch.no_grad()
def scores(model, loader, dev):
    """テストの特徴を先に全部 GPU に載せてから、モデルだけの時間を測る（データの合成は時間に入れない）"""
    model.eval()
    feats, ys = [], []
    for x, y in loader:
        feats.append(features(x, dev))
        ys.append(y)
    model(feats[0])                                  # ウォームアップ（1回目だけの準備の時間を測らない）
    sync(dev)
    t0 = time.perf_counter()
    out = [((model(s) - s) ** 2).mean(dim=(1, 2, 3)) for s in feats]   # 1サンプルの平均二乗誤差＝異常の得点
    sync(dev)
    infer_s = time.perf_counter() - t0
    return torch.cat(out).cpu().numpy(), torch.cat(ys).numpy(), infer_s


def run_one(a, seed, dev):
    set_seed(seed)
    normal_idx = list(range(0, 2 * a.n_train, 2))            # SynthMotor は偶数番が正常
    train = Subset(SynthMotor(2 * a.n_train, a.length, seed=1000 + seed), normal_idx)
    test = SynthMotor(a.n_test, a.length, seed=5000)          # 正常・異常が半分ずつ。種によらず同じ
    kw = dict(num_workers=a.workers, persistent_workers=a.workers > 0)
    tl = DataLoader(train, batch_size=a.batch, shuffle=True, drop_last=True, **kw)
    vl = DataLoader(test, batch_size=256, **kw)
    model = ConvAE().to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=a.lr)
    step_t, losses = [], []
    sync(dev)
    t_start = time.perf_counter()
    for ep in range(a.epochs):
        model.train()
        for x, _ in tl:
            t0 = time.perf_counter()
            s = features(x, dev)
            loss = ((model(s) - s) ** 2).mean()
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            sync(dev)
            step_t.append(time.perf_counter() - t0)
        losses.append(loss.item())
        print(f"  seed {seed} epoch {ep + 1}/{a.epochs} loss {losses[-1]:.4f}", flush=True)
    sync(dev)
    train_s = time.perf_counter() - t_start
    sc, y, infer_s = scores(model, vl, dev)
    auc = roc_auc(sc[y == 0], sc[y == 1])
    lo, hi = auc_boot_ci(sc[y == 0], sc[y == 1])
    med = float(np.median(step_t[10:])) if len(step_t) > 20 else float(np.median(step_t))
    print(f"  seed {seed}：学習 {train_s:.1f} 秒（1ステップの中央値 {1e3 * med:.1f} ms）、"
          f"ROC-AUC {auc:.3f}［{lo:.3f}, {hi:.3f}］、推論 {a.n_test / infer_s:.0f} 個/秒")
    return {"seed": seed, "train_s": train_s, "step_med_s": med, "auc": auc, "auc_lo": lo, "auc_hi": hi,
            "infer_per_s": a.n_test / infer_s, "epoch_loss": losses}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n-train", type=int, default=8000, help="正常の学習データの数")
    ap.add_argument("--n-test", type=int, default=2000)
    ap.add_argument("--length", type=int, default=4096)
    ap.add_argument("--epochs", type=int, default=5)
    ap.add_argument("--batch", type=int, default=64)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--seeds", type=int, nargs="+", default=[0])
    ap.add_argument("--device", default="auto")
    ap.add_argument("--label", default="run")
    ap.add_argument("--outdir", default="runs/p3c6")
    ap.add_argument("--price-usd-h", type=float, default=0.0, help="時間単価（Mac なら 0）")
    ap.add_argument("--usd-jpy", type=float, default=150.0)
    ap.add_argument("--quick", action="store_true", help="小さく速く動作確認する")
    a = ap.parse_args()
    if a.quick:
        a.n_train, a.n_test, a.epochs, a.label = 1000, 400, 2, a.label + "_quick"
    dev = pick_device(a.device)
    print(f"device {dev}、正常 {a.n_train} 個で {a.epochs} エポック、種 {a.seeds}")
    runs = [run_one(a, s, dev) for s in a.seeds]
    t = [r["train_s"] for r in runs]
    m, lo, hi = mean_ci(t)
    yen = [x / 3600 * a.price_usd_h * a.usd_jpy for x in t]
    am, alo, ahi = mean_ci([r["auc"] for r in runs], 0.0, 1.0)
    print(f"学習時間：平均 {m:.1f} 秒［95% 区間 {lo:.1f}, {hi:.1f}］（種 {len(t)} 個）")
    print(f"費用（学習だけ）：平均 {np.mean(yen):.2f} 円、ROC-AUC 平均 {am:.3f}［{alo:.3f}, {ahi:.3f}］")
    write_json(f"{a.outdir}/ae_{a.label}.json",
               {"task": "anomaly_ae", "args": vars(a), "env": env_summary(dev), "runs": runs,
                "train_s_mean_ci": [m, lo, hi], "auc_mean_ci": [am, alo, ahi], "yen_train": yen})


if __name__ == "__main__":
    main()

In [ ]:
%%writefile unet_seg.py
#!/usr/bin/env python3
"""III-6 課題2：U-Net によるセグメンテーション（粒子の領域を画素ごとに当てる）
--data synth：SEM 画像に似せた合成画像（明るい粒子＋背景のむら＋ぼけ＋雑音）。ダウンロード不要
--data npy  ：公開データセットを images.npy（N, H, W、0〜1 の float32）と masks.npy（N, H, W、0/1 の uint8）
              に変換したフォルダ（--npy-dir）。H・W は 16 の倍数にそろえておく
学習曲線（エポックごとの損失と検証の Dice）、Dice 係数、推論速度（枚/秒）を JSON に残す。
使い方:
  uv run python unet_seg.py --quick                         # Mac で動作確認
  uv run python unet_seg.py --seeds 0 1 2 --label mac       # Mac で本番と同じ設定
  python unet_seg.py --seeds 0 1 2 --label cuda --outdir /workspace/runs/p3c6 --price-usd-h 0.40
"""
import argparse
import os
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

from p3c6_util import env_summary, mean_ci, pick_device, sync, write_json
from train_toy import set_seed


def synth_image(rng, size):
    """粒子（楕円）を 3〜12 個置いた画像と正解マスク。どちらも (size, size) の float32"""
    yy, xx = np.mgrid[0:size, 0:size].astype(np.float32)
    mask = np.zeros((size, size), np.float32)
    for _ in range(int(rng.integers(3, 13))):
        cy, cx = rng.uniform(0, size, 2)
        ry, rx = rng.uniform(4, size / 8, 2)
        th = rng.uniform(0, np.pi)
        dy, dx = yy - cy, xx - cx
        u = (dx * np.cos(th) + dy * np.sin(th)) / rx
        v = (-dx * np.sin(th) + dy * np.cos(th)) / ry
        mask[u ** 2 + v ** 2 <= 1] = 1
    g = rng.uniform(-0.3, 0.3, 2)                                     # 背景のむら（明るさの傾き）
    img = 0.3 + g[0] * yy / size + g[1] * xx / size + 0.35 * mask
    k = np.ones(5, np.float32) / 5                                    # 5×5 の平均でぼかす（縦・横）
    img = np.apply_along_axis(lambda r: np.convolve(r, k, "same"), 1, img)
    img = np.apply_along_axis(lambda c: np.convolve(c, k, "same"), 0, img)
    img = img + rng.normal(0, 0.12, img.shape)                        # 雑音（SEM のざらつきの代わり）
    return np.clip(img, 0, 1).astype(np.float32), mask


class SynthParticles(Dataset):
    def __init__(self, n, size, seed):
        self.n, self.size, self.seed = n, size, seed

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        img, mask = synth_image(np.random.default_rng([self.seed, i]), self.size)
        return torch.from_numpy(img)[None], torch.from_numpy(mask)[None]


class NpyPairs(Dataset):
    """images.npy・masks.npy を memmap で読む（III-5 の CachedMotor と同じく、ワーカーの中で開く）"""
    def __init__(self, folder, idx):
        self.folder, self.idx, self.img, self.msk = folder, idx, None, None

    def __len__(self):
        return len(self.idx)

    def __getitem__(self, i):
        if self.img is None:
            self.img = np.load(os.path.join(self.folder, "images.npy"), mmap_mode="r")
            self.msk = np.load(os.path.join(self.folder, "masks.npy"), mmap_mode="r")
        j = self.idx[i]
        return (torch.from_numpy(np.array(self.img[j], np.float32))[None],
                torch.from_numpy(np.array(self.msk[j], np.float32))[None])


def block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                         nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True))


class UNet(nn.Module):
    """4段の U-Net（16→32→64→128、底 256）。出力はロジット（シグモイドの前）"""
    def __init__(self, w=16):
        super().__init__()
        ch = [w, 2 * w, 4 * w, 8 * w]
        self.downs = nn.ModuleList()
        cin = 1
        for c in ch:
            self.downs.append(block(cin, c))
            cin = c
        self.bottom = block(ch[-1], 2 * ch[-1])
        self.ups, self.dec = nn.ModuleList(), nn.ModuleList()
        cin = 2 * ch[-1]
        for c in reversed(ch):
            self.ups.append(nn.ConvTranspose2d(cin, c, 2, stride=2))
            self.dec.append(block(2 * c, c))                         # スキップ接続と連結するので 2c
            cin = c
        self.out = nn.Conv2d(ch[0], 1, 1)

    def forward(self, x):
        skips = []
        for d in self.downs:
            x = d(x)
            skips.append(x)
            x = F.max_pool2d(x, 2)
        x = self.bottom(x)
        for up, dec, s in zip(self.ups, self.dec, reversed(skips)):
            x = dec(torch.cat([up(x), s], dim=1))
        return self.out(x)


def dice_loss(logit, m, eps=1.0):
    p = torch.sigmoid(logit)
    inter = (p * m).sum(dim=(1, 2, 3))
    return (1 - (2 * inter + eps) / (p.sum(dim=(1, 2, 3)) + m.sum(dim=(1, 2, 3)) + eps)).mean()


@torch.no_grad()
def evaluate(model, loader, dev):
    """画像ごとの Dice（しきい値 0.5）を返す。どちらも空の画像は 1 とする"""
    model.eval()
    out = []
    for x, m in loader:
        x, m = x.to(dev), m.to(dev)
        p = (torch.sigmoid(model(x)) > 0.5).float()
        inter = (p * m).sum(dim=(1, 2, 3))
        den = p.sum(dim=(1, 2, 3)) + m.sum(dim=(1, 2, 3))
        out.append(torch.where(den > 0, 2 * inter / den.clamp(min=1), torch.ones_like(den)).cpu())
    return torch.cat(out).numpy()


@torch.no_grad()
def infer_speed(model, dev, h, w, batch, reps=10):
    """データの読み込みを含まない、モデルだけの推論の速さ（枚/秒）"""
    model.eval()
    x = torch.rand(batch, 1, h, w, device=dev)
    for _ in range(3):
        model(x)
    sync(dev)
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        model(x)
        sync(dev)
        ts.append(time.perf_counter() - t0)
    return batch / float(np.median(ts))


def make_sets(a, seed):
    if a.data == "synth":
        return SynthParticles(a.n_train, a.size, 100 + seed), SynthParticles(a.n_val, a.size, 9000)
    n = np.load(os.path.join(a.npy_dir, "masks.npy"), mmap_mode="r").shape[0]
    perm = np.random.default_rng(0).permutation(n)                   # 分け方は種 0 で固定（全ての反復で同じ）
    nv = max(1, n // 5)
    return NpyPairs(a.npy_dir, perm[nv:]), NpyPairs(a.npy_dir, perm[:nv])


def run_one(a, seed, dev):
    set_seed(seed)
    tr, va = make_sets(a, seed)
    kw = dict(num_workers=a.workers, persistent_workers=a.workers > 0)
    tl = DataLoader(tr, batch_size=a.batch, shuffle=True, drop_last=True, **kw)
    vl = DataLoader(va, batch_size=32, **kw)
    model = UNet().to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=a.lr)
    curve = []
    sync(dev)
    t_start = time.perf_counter()
    for ep in range(a.epochs):
        model.train()
        tot, nb = 0.0, 0
        for x, m in tl:
            x, m = x.to(dev, non_blocking=True), m.to(dev, non_blocking=True)
            logit = model(x)
            loss = F.binary_cross_entropy_with_logits(logit, m) + dice_loss(logit, m)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            tot, nb = tot + loss.item(), nb + 1
        d = evaluate(model, vl, dev)
        curve.append({"epoch": ep + 1, "train_loss": tot / nb, "val_dice": float(d.mean()),
                      "elapsed_s": time.perf_counter() - t_start})
        print(f"  seed {seed} epoch {ep + 1}/{a.epochs} 損失 {tot / nb:.3f}  検証 Dice {d.mean():.3f}", flush=True)
    train_s = time.perf_counter() - t_start
    d = evaluate(model, vl, dev)
    h, w = va[0][0].shape[-2:]                                     # npy のときは画像の形から
    ips = infer_speed(model, dev, h, w, 16)
    print(f"  seed {seed}：学習 {train_s:.1f} 秒（エポックごとの検証を含む）、Dice 平均 {d.mean():.3f}"
          f"（中央値 {np.median(d):.3f}、最小 {d.min():.3f}）、推論 {ips:.0f} 枚/秒（{h}×{w}、バッチ 16）")
    return {"seed": seed, "train_s": train_s, "dice_mean": float(d.mean()), "dice_median": float(np.median(d)),
            "dice_min": float(d.min()), "infer_img_per_s": ips, "curve": curve}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", choices=["synth", "npy"], default="synth")
    ap.add_argument("--npy-dir", default="data/seg_public")
    ap.add_argument("--size", type=int, default=128, help="synth の画像の一辺（16 の倍数）")
    ap.add_argument("--n-train", type=int, default=2000)
    ap.add_argument("--n-val", type=int, default=200)
    ap.add_argument("--epochs", type=int, default=6)
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--seeds", type=int, nargs="+", default=[0])
    ap.add_argument("--device", default="auto")
    ap.add_argument("--label", default="run")
    ap.add_argument("--outdir", default="runs/p3c6")
    ap.add_argument("--price-usd-h", type=float, default=0.0)
    ap.add_argument("--usd-jpy", type=float, default=150.0)
    ap.add_argument("--quick", action="store_true")
    a = ap.parse_args()
    if a.quick:
        a.n_train, a.n_val, a.epochs, a.label = 320, 64, 2, a.label + "_quick"
    dev = pick_device(a.device)
    print(f"device {dev}、data {a.data}、{a.epochs} エポック、種 {a.seeds}")
    runs = [run_one(a, s, dev) for s in a.seeds]
    m, lo, hi = mean_ci([r["train_s"] for r in runs])
    dm, dlo, dhi = mean_ci([r["dice_mean"] for r in runs], 0.0, 1.0)
    yen = [r["train_s"] / 3600 * a.price_usd_h * a.usd_jpy for r in runs]
    print(f"学習時間：平均 {m:.1f} 秒［95% 区間 {lo:.1f}, {hi:.1f}］、Dice：平均 {dm:.3f}［{dlo:.3f}, {dhi:.3f}］"
          f"（種 {len(runs)} 個）、費用 平均 {np.mean(yen):.2f} 円")
    write_json(f"{a.outdir}/unet_{a.label}.json",
               {"task": "unet_seg", "args": vars(a), "env": env_summary(dev), "runs": runs,
                "train_s_mean_ci": [m, lo, hi], "dice_mean_ci": [dm, dlo, dhi], "yen_train": yen})


if __name__ == "__main__":
    main()

In [ ]:
%%writefile nmf_torch.py
#!/usr/bin/env python3
"""III-6 課題3：NMF の GPU 実装と、CPU との損益分岐点
第7章の nmf()（NumPy、補助関数法の乗法更新）と同じ式を PyTorch で書き、
(1) 同じ初期値から NumPy 版と数値で照合し、(2) 行列の大きさを変えて 1 反復の時間を測る。
~/home-cloud は librosa を入れていないので、第7章の nmf() の式をここに写した（nmf_numpy）。
使い方:
  uv run python nmf_torch.py check                          # NumPy 版との照合（float64 の CPU と、GPU の float32）
  uv run python nmf_torch.py bench --label mac              # 大きさを変えて時間を測る
  python nmf_torch.py bench --label cuda --outdir /workspace/runs/p3c6
"""
import argparse
import time

import numpy as np
import torch

from p3c6_util import bench, env_summary, pick_device, write_json

EPS = 1e-12


def init_wh(X, k, seed):
    """第7章と同じ初期値（NumPy の乱数）。torch 版にも同じものを渡して照合できるようにする"""
    rng = np.random.default_rng(seed)
    F_, T = X.shape
    return rng.uniform(0.1, 1.0, (F_, k)) * X.mean(), rng.uniform(0.1, 1.0, (k, T))


def nmf_numpy(X, k, div="kl", n_iter=200, seed=0):
    """第7章 ch7/nmf.py の nmf() と同じ計算（コストの履歴は時間を測るため省いた）"""
    W, H = init_wh(X, k, seed)
    for _ in range(n_iter):
        Y = W @ H + EPS
        if div == "kl":
            W *= ((X / Y) @ H.T) / (H.sum(axis=1)[None, :] + EPS)
            Y = W @ H + EPS
            H *= (W.T @ (X / Y)) / (W.sum(axis=0)[:, None] + EPS)
        else:
            W *= np.sqrt(((X / Y ** 2) @ H.T) / ((1 / Y) @ H.T + EPS))
            Y = W @ H + EPS
            H *= np.sqrt((W.T @ (X / Y ** 2)) / (W.T @ (1 / Y) + EPS))
        s = W.sum(axis=0) + EPS
        W /= s
        H *= s[:, None]
    return W, H


def nmf_torch(X, k, div="kl", n_iter=200, seed=0, device="cpu", dtype=torch.float32, W0=None, H0=None):
    """nmf_numpy と同じ更新式。X は NumPy でも torch でもよい。戻り値は torch の W, H（device の上）"""
    dev = torch.device(device)
    X = torch.as_tensor(X, dtype=dtype, device=dev)
    if W0 is None:
        W0, H0 = init_wh(X.cpu().double().numpy(), k, seed)
    W = torch.as_tensor(W0, dtype=dtype, device=dev).clone()
    H = torch.as_tensor(H0, dtype=dtype, device=dev).clone()
    eps = EPS if dtype == torch.float64 else 1e-12     # float32 でも 1e-12 は表せる（最小の正規数 約 1e-38）
    for _ in range(n_iter):
        Y = W @ H + eps
        if div == "kl":
            W *= ((X / Y) @ H.T) / (H.sum(dim=1)[None, :] + eps)
            Y = W @ H + eps
            H *= (W.T @ (X / Y)) / (W.sum(dim=0)[:, None] + eps)
        else:
            W *= torch.sqrt(((X / Y ** 2) @ H.T) / ((1 / Y) @ H.T + eps))
            Y = W @ H + eps
            H *= torch.sqrt((W.T @ (X / Y ** 2)) / (W.T @ (1 / Y) + eps))
        s = W.sum(dim=0) + eps
        W /= s
        H *= s[:, None]
    return W, H


def test_matrix(F_, T, k_true=6, seed=0):
    """非負の低ランク行列＋小さな雑音（照合と速さの測定用。音でなくてよい）"""
    rng = np.random.default_rng(seed)
    return rng.gamma(1.0, 1.0, (F_, k_true)) @ rng.gamma(0.5, 1.0, (k_true, T)) + rng.uniform(0, 0.01, (F_, T))


def rel(a, b):
    return float(np.linalg.norm(a - b) / np.linalg.norm(b))


def check(a):
    X = test_matrix(257, 400)
    dev = pick_device(a.device)
    rows = []
    for div in ("kl", "is"):
        Xd = X if div == "kl" else X ** 2                # KL は振幅、IS はパワー（第7章と同じ使い分け）
        Wn, Hn = nmf_numpy(Xd, 8, div, a.iters, 0)
        Yn = Wn @ Hn
        for name, d, dt in (("cpu float64", "cpu", torch.float64), ("cpu float32", "cpu", torch.float32),
                            (f"{dev.type} float32", dev.type, torch.float32)):
            W, H = nmf_torch(Xd, 8, div, a.iters, 0, d, dt)
            W, H = W.cpu().double().numpy(), H.cpu().double().numpy()
            r = {"div": div, "impl": name, "rel_W": rel(W, Wn), "rel_H": rel(H, Hn), "rel_WH": rel(W @ H, Yn)}
            rows.append(r)
            print(f"{div.upper()} {name:13s}：W の相対差 {r['rel_W']:.1e}、H {r['rel_H']:.1e}、WH {r['rel_WH']:.1e}")
    print("合格の目安：float64 は 1e-10 以下、float32 は WH が 1e-3 以下（W・H は反復で差が広がることがある）")
    write_json(f"{a.outdir}/nmf_check_{a.label}.json", {"iters": a.iters, "rows": rows, "env": env_summary(dev)})


def bench_cmd(a):
    dev = pick_device(a.device)
    sizes = [(257, 250), (513, 500), (1025, 1000), (1025, 4000), (2049, 8000)]
    if a.max_size:
        sizes = sizes[:a.max_size]
    rows = []
    for F_, T in sizes:
        X = test_matrix(F_, T)
        n = a.iters
        r = {"F": F_, "T": T, "k": a.k, "iters": n}
        t = bench(lambda: nmf_numpy(X, a.k, "kl", n, 0), torch.device("cpu"), reps=a.reps, warmup=1)
        r["numpy_s_per_iter"] = [x / n for x in t]
        Xt = torch.as_tensor(X, dtype=torch.float32, device=dev)
        W0, H0 = init_wh(X, a.k, 0)
        t = bench(lambda: nmf_torch(Xt, a.k, "kl", n, 0, dev.type, torch.float32, W0, H0), dev, reps=a.reps, warmup=1)
        r["gpu_s_per_iter"] = [x / n for x in t]
        Xc = torch.as_tensor(X, dtype=torch.float32)
        t = bench(lambda: nmf_torch(Xc, a.k, "kl", n, 0, "cpu", torch.float32, W0, H0), torch.device("cpu"),
                  reps=a.reps, warmup=1)
        r["torchcpu_s_per_iter"] = [x / n for x in t]
        med = {key: 1e3 * float(np.median(r[key])) for key in ("numpy_s_per_iter", "torchcpu_s_per_iter", "gpu_s_per_iter")}
        r["speedup_vs_numpy"] = med["numpy_s_per_iter"] / med["gpu_s_per_iter"]
        rows.append(r)
        print(f"{F_:5d}×{T:5d}（要素 {F_ * T / 1e6:6.2f} M）：NumPy {med['numpy_s_per_iter']:8.2f} ms、"
              f"torch CPU f32 {med['torchcpu_s_per_iter']:8.2f} ms、{dev.type} f32 {med['gpu_s_per_iter']:8.2f} ms"
              f"／反復 → NumPy 比 {r['speedup_vs_numpy']:.1f} 倍", flush=True)
    write_json(f"{a.outdir}/nmf_bench_{a.label}.json", {"rows": rows, "env": env_summary(dev)})


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["check", "bench"])
    ap.add_argument("--iters", type=int, default=None, help="check は 200、bench は 20（1回の測定の反復数）")
    ap.add_argument("--k", type=int, default=16)
    ap.add_argument("--reps", type=int, default=5)
    ap.add_argument("--max-size", type=int, default=0, help="大きさの表の先頭から何個まで（0 は全部）")
    ap.add_argument("--device", default="auto")
    ap.add_argument("--label", default="run")
    ap.add_argument("--outdir", default="runs/p3c6")
    a = ap.parse_args()
    if a.iters is None:
        a.iters = 200 if a.cmd == "check" else 20
    check(a) if a.cmd == "check" else bench_cmd(a)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile string_fd.py
#!/usr/bin/env python3
"""III-6 課題4：弦の振動（1次元の波動方程式）を差分法で解き、GPU が得になる規模を測る
  ∂²u/∂t² = c² ∂²u/∂x²、両端固定 u(0,t)=u(L,t)=0、初速 0
中心差分（リープフロッグ）：u[n+1] = 2u[n] − u[n−1] + r²(u[i+1] − 2u[i] + u[i−1])、r = cΔt/Δx（クーラン数）
r ≤ 1（CFL 条件）なら安定。B 本の弦（張力・初期形の違う弦）をまとめて (B, N) の配列で一度に進める。
使い方:
  uv run python string_fd.py check                    # 解析解との照合・収束の次数・CFL 条件
  uv run python string_fd.py bench --label mac        # 規模（B×N）を変えて NumPy・torch CPU・GPU を比べる
  python string_fd.py bench --label cuda --outdir /workspace/runs/p3c6
"""
import argparse
import math

import numpy as np
import torch

from p3c6_util import bench, env_summary, pick_device, write_json

L_STR, C = 0.65, 200.0          # 弦の長さ 0.65 m（ギターくらい）、波の速さ 200 m/s（基本振動 約 154 Hz）


def exact(x, t, modes):
    """初期形 Σ a_m sin(mπx/L)、初速 0 の解析解：Σ a_m sin(mπx/L) cos(mπct/L)"""
    return sum(a * np.sin(m * np.pi * x / L_STR) * np.cos(m * np.pi * C * t / L_STR) for m, a in modes)


def simulate(u0, r, n_steps, xp):
    """u0：(B, N) の初期形（両端は 0）。xp は np か torch。n_steps 進めた u を返す"""
    lap = lambda u: u[:, 2:] - 2 * u[:, 1:-1] + u[:, :-2]
    u_prev = u0
    u = u0.clone() if xp is torch else u0.copy()
    u[:, 1:-1] = u0[:, 1:-1] + 0.5 * r * r * lap(u0)   # 初速 0 の最初の1歩（テイラー展開の2次まで）
    for _ in range(n_steps - 1):
        u_next = 2 * u - u_prev
        u_next[:, 1:-1] += r * r * lap(u)
        u_next[:, 0] = 0
        u_next[:, -1] = 0
        u_prev, u = u, u_next
    return u


def check(a):
    modes = [(1, 1.0), (3, 0.3)]
    rows = []
    print("(1) 格子を細かくしたときの誤差（r = 0.5、t = 0.01 s まで。2次の方法なら N を 2 倍で誤差は約 1/4）")
    for N in (51, 101, 201, 401):
        x = np.linspace(0, L_STR, N)
        dx = x[1] - x[0]
        r = 0.5
        dt = r * dx / C
        n = int(round(0.01 / dt))
        u = simulate(exact(x, 0, modes)[None, :].copy(), r, n, np)
        err = float(np.max(np.abs(u[0] - exact(x, n * dt, modes))))
        rows.append({"N": N, "r": r, "steps": n, "t": n * dt, "max_err": err})
        print(f"  N = {N:4d}：Δx = {1e3 * dx:6.2f} mm、Δt = {1e6 * dt:6.2f} µs、{n:5d} 歩、最大誤差 {err:.2e}")
    print("(2) クーラン数 r を変える（N = 201、0.01 s まで）")
    x = np.linspace(0, L_STR, 201)
    dx = x[1] - x[0]
    for r in (0.5, 0.9, 1.0, 1.02, 1.1):
        dt = r * dx / C
        n = int(round(0.01 / dt))
        u0 = exact(x, 0, modes)[None, :].copy()
        u0[0, 100] += 1e-9                       # 丸め誤差の代わりの、ごく小さな乱れ
        u = simulate(u0, r, n, np)
        err = float(np.max(np.abs(u[0] - exact(x, n * dt, modes))))
        rows.append({"N": 201, "r": r, "steps": n, "t": n * dt, "max_err": err})
        print(f"  r = {r:4.2f}：{n:4d} 歩、最大誤差 {err:.2e}{'  ← 発散（CFL 条件 r ≤ 1 を破った）' if err > 1 else ''}")
    dev = pick_device(a.device)
    x = np.linspace(0, L_STR, 201)
    u0 = np.tile(exact(x, 0, modes), (4, 1))
    un = simulate(u0.copy(), 0.9, 500, np)
    ut = simulate(torch.tensor(u0, dtype=torch.float32, device=dev), 0.9, 500, torch).cpu().numpy()
    d = float(np.max(np.abs(un - ut)))
    print(f"(3) NumPy（float64）と {dev.type}（float32）の差（N = 201、500 歩）：最大 {d:.1e}（目安 1e-4 以下）")
    write_json(f"{a.outdir}/string_check_{a.label}.json", {"rows": rows, "np_vs_gpu_max": d, "env": env_summary(dev)})


def bench_cmd(a):
    dev = pick_device(a.device)
    shapes = [(1, 1001), (16, 1001), (64, 4001), (256, 4001), (1024, 4001), (1024, 16001)]
    if a.max_size:
        shapes = shapes[:a.max_size]
    rows = []
    for B, N in shapes:
        x = np.linspace(0, L_STR, N)
        u0 = np.sin(np.pi * x / L_STR)[None, :] * np.linspace(0.5, 1.5, B)[:, None]
        r = {"B": B, "N": N, "steps": a.steps}
        t = bench(lambda: simulate(u0.astype(np.float32), 0.9, a.steps, np), torch.device("cpu"), a.reps, 1)
        r["numpy_s"] = t
        uc = torch.tensor(u0, dtype=torch.float32)
        r["torchcpu_s"] = bench(lambda: simulate(uc, 0.9, a.steps, torch), torch.device("cpu"), a.reps, 1)
        ug = uc.to(dev)
        r["gpu_s"] = bench(lambda: simulate(ug, 0.9, a.steps, torch), dev, a.reps, 1)
        med = {k: 1e6 * float(np.median(r[k])) / a.steps for k in ("numpy_s", "torchcpu_s", "gpu_s")}
        r["speedup_vs_numpy"] = med["numpy_s"] / med["gpu_s"]
        rows.append(r)
        print(f"B = {B:5d}、N = {N:6d}（格子点 {B * N / 1e6:6.2f} M）：1歩 NumPy {med['numpy_s']:9.1f} µs、"
              f"torch CPU {med['torchcpu_s']:9.1f} µs、{dev.type} {med['gpu_s']:9.1f} µs → NumPy 比 "
              f"{r['speedup_vs_numpy']:.2f} 倍", flush=True)
    write_json(f"{a.outdir}/string_bench_{a.label}.json", {"rows": rows, "env": env_summary(dev)})


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["check", "bench"])
    ap.add_argument("--steps", type=int, default=200)
    ap.add_argument("--reps", type=int, default=5)
    ap.add_argument("--max-size", type=int, default=0)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--label", default="run")
    ap.add_argument("--outdir", default="runs/p3c6")
    a = ap.parse_args()
    check(a) if a.cmd == "check" else bench_cmd(a)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile p3c6_table.py
#!/usr/bin/env python3
"""III-6：Mac とクラウドの結果（runs/p3c6/・runs/p3c6_vast_m1/ など、マシンごとのフォルダの JSON）を1つの表にまとめる
  学習の課題（ae_*.json・unet_*.json）：学習時間の平均と 95% 区間（種が反復の単位）、精度、費用
  計算の課題（nmf_bench_*.json・string_bench_*.json）：大きさごとの中央値と、GPU が NumPy に勝つ最小の大きさ
使い方: uv run python p3c6_table.py runs/p3c6 runs/p3c6_vast_m1 runs/p3c6_vast_m2
"""
import glob
import json
import os
import sys

import numpy as np

from p3c6_util import mean_ci


def load(dirs, pattern):
    out = []
    for d in dirs:
        for p in sorted(glob.glob(os.path.join(d, pattern))):
            if "_quick" in p:
                continue
            with open(p) as f:
                out.append((p, json.load(f)))
    return out


def train_table(dirs, pattern, metric, name):
    rows = load(dirs, pattern)
    if not rows:
        return
    print(f"\n■ {name}（学習時間は種ごとの値の平均と 95% t 区間）")
    print(f"{'ファイル':34s} {'machine':9s} {'device':6s} {'種':>3s} {'学習 s 平均':>10s} {'95% 区間':>17s} {metric:>8s} {'円/回':>7s}")
    for p, j in rows:
        t = [r["train_s"] for r in j["runs"]]
        m, lo, hi = mean_ci(t)
        acc = np.mean([r[metric] for r in j["runs"]])
        print(f"{os.path.relpath(p):34s} {j['env'].get('machine', '?'):9s} {j['env']['device']:6s} {len(t):3d} {m:10.1f} "
              f"[{lo:7.1f}, {hi:7.1f}] {acc:8.3f} {np.mean(j['yen_train']):7.2f}")


def bench_table(dirs, pattern, size_keys, name, unit, scale):
    rows = load(dirs, pattern)
    for p, j in rows:
        print(f"\n■ {name}：{os.path.relpath(p)}（{j['env'].get('machine', '?')}、{j['env']['device_name']}、中央値 {unit}）")
        first_win = None
        for r in j["rows"]:
            per = r.get("iters") or r.get("steps")
            med = {k: scale * float(np.median(r[k])) / (per if "iters" not in r else 1)
                   for k in r if k.endswith("_s") or k.endswith("_per_iter")}
            size = "×".join(str(r[k]) for k in size_keys)
            win = r["speedup_vs_numpy"] > 1
            if win and first_win is None:
                first_win = size
            print(f"  {size:>12s}： " + "  ".join(f"{k.split('_')[0]} {v:9.2f}" for k, v in med.items())
                  + f"  → NumPy÷GPU {r['speedup_vs_numpy']:6.2f} 倍{'（GPU が得）' if win else ''}")
        print(f"  GPU が NumPy より速くなった最小の大きさ：{first_win or 'この範囲には無い'}")


def main():
    dirs = sys.argv[1:] or ["runs/p3c6"]
    train_table(dirs, "ae_*.json", "auc", "課題1 異常検知（ROC-AUC）")
    train_table(dirs, "unet_*.json", "dice_mean", "課題2 U-Net（Dice）")
    bench_table(dirs, "nmf_bench_*.json", ("F", "T"), "課題3 NMF 1反復", "ms", 1e3)
    bench_table(dirs, "string_bench_*.json", ("B", "N"), "課題4 弦 1歩", "µs", 1e6)


if __name__ == "__main__":
    main()

In [ ]:
# MACHINE・label・結果のフォルダを決め、run()・ok()・load() を用意する（ノートブック版で足したもの）
import os, sys, json, math, shlex, subprocess
import numpy as np
import torch
sys.path.insert(0, ".")
import p3c6_util
from train_toy import pick_device, device_name

IN_COLAB = "google.colab" in sys.modules
os.environ["GIT_CEILING_DIRECTORIES"] = os.path.dirname(os.path.abspath("."))   # git_info() が、このフォルダより上の Git のリポジトリ（教科書のフォルダなど）のコミット ID を拾わないように
DEV = pick_device()
if DEV.type == "cuda":
    gpu = device_name(DEV)                                    # 例："Tesla T4"
    short = next((k for k in ("T4", "L4", "A100", "H100", "V100", "P100", "A10") if k in gpu), "gpu").lower()
    MACHINE, LABEL = f"colab_{short}" if IN_COLAB else f"local_{short}", f"cuda_{short}"
elif DEV.type == "mps":
    MACHINE, LABEL = "mac", "mac_mps"
else:
    MACHINE, LABEL = ("colab_cpu" if IN_COLAB else "cpu"), "cpu"
# MACHINE = "colab_t4_2"                                      # 別の日に別のマシンで測るときは、名前を変えて上書きを避ける
os.environ["MACHINE"] = MACHINE                               # 教科書の export MACHINE=…（p3c6_util.env_summary() が読む）
R = "runs/p3c6" if MACHINE == "mac" else f"runs/p3c6_{MACHINE}"
LABEL_CPU = f"{MACHINE}_cpu"                                  # 教科書の (3) の mac_cpu
PRICE = 0.0           # USD/時。Colab の無料の枠なら 0。有料なら「1 時間のユニット数 × 1 ユニットの USD」（公式サイトで要確認）
USD_JPY = 150.0       # 円/USD（教科書と同じ仮の値）
os.makedirs(R, exist_ok=True)
print(f"device {DEV}（{device_name(DEV)}）、MACHINE={MACHINE}、label={LABEL}、結果は {R}/、CPU のコア {os.cpu_count()} 個")
if DEV.type == "cuda":
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"])   # 教科書の手順6 で記録する GPU の名前


def run(cmd):
    """ターミナルの `uv run --locked python <cmd>` の代わり。別のプロセスで実行し、出力をそのまま表示して返す"""
    print("$ python", cmd, flush=True)
    p = subprocess.Popen([sys.executable, "-u", *shlex.split(cmd)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, env=dict(os.environ))
    lines = []
    for line in p.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    rc = p.wait()
    assert rc == 0, f"エラーで終わりました（returncode {rc}）。上の表示を見てください"
    return "".join(lines)


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def load(path):
    with open(path) as f:
        return json.load(f)


FILES = [n for n in ("train_toy.py", "runlog.py", "prep_cache.py", "train_fast.py", "p3c6_util.py", "anomaly_ae.py",
                     "unet_seg.py", "nmf_torch.py", "string_fd.py", "p3c6_table.py")]

In [ ]:
# 確かめ（教科書 III-6-5 の手順の「確かめること」。試しの設定でも本番でも成り立つことだけ）
ok(all(os.path.exists(f) for f in FILES), f"{len(FILES)} 個のファイルを書き出した")
r = subprocess.run([sys.executable, "-c", "import p3c6_util"], capture_output=True, text=True)
ok(r.returncode == 0, "python -c \"import p3c6_util\" がエラーなく終わる（手順1。runlog・train_fast・train_toy・prep_cache も読めた）" + r.stderr[-300:])
g = p3c6_util.git_info()
ok("commit" in g, f"git_info() が落ちずに値を返す（{g['commit'][:12]}。Git のフォルダの外や Git が無いときは unknown）")

## 1. 課題1：正常だけで学ぶ異常検知（III-6-3-3、`anomaly_ae.py`）

III-2 の合成モータ音 `SynthMotor` の**正常（偶数番）だけ**でオートエンコーダを学習し、作り直しの誤差（平均二乗誤差）を異常の得点にして ROC-AUC を出します。

まず手順 2 の動作確認（`--quick`：正常 1000 個、2 エポック。1 分ほど）です。結果は `{R}/ae_{LABEL}_quick.json` に入ります（`_quick` の付いたファイルは、5 節の表には入りません）。

In [ ]:
run(f"anomaly_ae.py --quick --label {LABEL} --outdir {R}")

In [ ]:
# 確かめ（教科書 III-6-5 の手順の「確かめること」。試しの設定でも本番でも成り立つことだけ）
j = load(f"{R}/ae_{LABEL}_quick.json")
r0 = j["runs"][0]
ok(r0["auc"] >= 0.9, f"[手順2] --quick の ROC-AUC が 0.9 以上（{r0['auc']:.3f}）")
ok(r0["auc_lo"] <= r0["auc"] <= r0["auc_hi"], f"ブートストラップの 95% 区間［{r0['auc_lo']:.3f}, {r0['auc_hi']:.3f}］が AUC を含む")
ok(len(r0["epoch_loss"]) == j["args"]["epochs"] and r0["train_s"] > 0 and r0["infer_per_s"] > 0, "エポックごとの損失・学習時間・推論の速さが記録された")
ok(len(j["runs"]) >= 2 or all(math.isnan(v) for v in j["train_s_mean_ci"][1:]), "種が 1 個なら学習時間の区間は nan（III-6-7 の表の「区間が [nan, nan]」）")
ok(j["env"]["machine"] == MACHINE and j["env"]["device"] == DEV.type and "commit" in j["env"]["git"],
   f"env に machine（{MACHINE}）・device（{DEV.type}）・git（{j['env']['git']['commit'][:12]}）が入った")

次に手順 4（本番と同じ設定を、種 0・1・2 の 3 回）です。**ノートブックの既定は小さい設定**（正常 1000 個 × 2 エポック、テスト 400 個。Mac で 1 分ほど）にしています。教科書の本番（正常 8000 個 × 5 エポック、テスト 2000 個。Mac の MPS で 1 回約 20 秒）にするときは、コメントのほうの行を使ってください。

学習時間の平均と 95% 区間（種が反復の単位）、AUC の平均と区間（0〜1 に切ったもの）、費用（学習だけ）が表示されます。種 0 の学習時間には、DataLoader のワーカーの起動など 1 回目だけの時間が入ることがあります（III-6-7）。

In [ ]:
run(f"anomaly_ae.py --seeds 0 1 2 --n-train 1000 --n-test 400 --epochs 2 --label {LABEL} --outdir {R} --price-usd-h {PRICE}")
# 本番（教科書の (2)。Colab なら教科書の「インスタンス｜コマンドのまとめ」と同じ）：
# run(f"anomaly_ae.py --seeds 0 1 2 --label {LABEL} --outdir {R} --price-usd-h {PRICE}")

In [ ]:
# 確かめ（教科書 III-6-5 の手順の「確かめること」。試しの設定でも本番でも成り立つことだけ）
j = load(f"{R}/ae_{LABEL}.json")
t = [r["train_s"] for r in j["runs"]]
m, lo, hi = j["train_s_mean_ci"]
ok(len(t) == len(j["args"]["seeds"]) >= 2 and abs(m - np.mean(t)) < 1e-9 and lo <= m <= hi,
   f"[手順4] 種 {len(t)} 個の学習時間の平均 {m:.1f} 秒と 95% 区間［{lo:.1f}, {hi:.1f}］")
h = p3c6_util.t975(len(t) - 1) * np.std(t, ddof=1) / math.sqrt(len(t))
ok(abs((hi - lo) / 2 - h) < 1e-9, f"区間の半分の幅が t_0.975 × s/√m（{h:.2f} 秒。III-6-3-2 の式）")
am, alo, ahi = j["auc_mean_ci"]
ok(0 <= alo <= am <= ahi <= 1, f"ROC-AUC の平均 {am:.3f}［{alo:.3f}, {ahi:.3f}］は 0〜1 に切ってある")
ok(all(abs(y - x / 3600 * PRICE * USD_JPY) < 1e-9 for x, y in zip(t, j["yen_train"])), f"費用（学習だけ）＝ 時間 ÷ 3600 × {PRICE} USD/時 × {USD_JPY} 円/USD")
ok(all(r["step_med_s"] > 0 for r in j["runs"]), "1 ステップの時間の中央値（最初の 10 ステップを除く）が記録された")

## 2. 課題2：U-Net による粒子の領域分割（III-6-3-4、`unet_seg.py`）

SEM 画像に少し似せた合成画像（楕円の粒子、背景の明るさの傾き、ぼけ、雑音）で U-Net を学習し、検証の Dice 係数を見ます。損失は BCE ＋ Dice 損失です。

まず手順 2 の動作確認（`--quick`：320 枚、2 エポック）です。1 エポック目の Dice は 0 に近いことがあり、2 エポック目で急に上がります。

In [ ]:
run(f"unet_seg.py --quick --label {LABEL} --outdir {R}")

In [ ]:
# 確かめ（教科書 III-6-5 の手順の「確かめること」。試しの設定でも本番でも成り立つことだけ）
j = load(f"{R}/unet_{LABEL}_quick.json")
r0 = j["runs"][0]
cv = [c["val_dice"] for c in r0["curve"]]
ok(len(cv) == j["args"]["epochs"] and cv[-1] >= 0.8, f"[手順2] U-Net の検証 Dice が 2 エポック目で 0.8 以上（エポックごと {[round(v, 3) for v in cv]}。1 エポック目は 0 に近いことがある）")
ok(0 <= r0["dice_min"] <= r0["dice_median"] <= 1 and 0 <= r0["dice_mean"] <= 1, f"Dice の最小 {r0['dice_min']:.3f} ≤ 中央値 {r0['dice_median']:.3f}（平均 {r0['dice_mean']:.3f}）")
ok(all(a["elapsed_s"] < b["elapsed_s"] for a, b in zip(r0["curve"], r0["curve"][1:])) and r0["infer_img_per_s"] > 0, "学習曲線（curve）と推論の速さが記録された")

手順 4（種 0・1・2）です。**既定は小さい設定**（320 枚 × 2 エポック、検証 64 枚）。教科書の本番は 2000 枚 × 6 エポック、検証 200 枚（Mac の MPS で 1 回約 70 秒）です。平均だけでなく、いちばん悪い画像の Dice（最小）も見てください。

In [ ]:
run(f"unet_seg.py --seeds 0 1 2 --n-train 320 --n-val 64 --epochs 2 --label {LABEL} --outdir {R} --price-usd-h {PRICE}")
# 本番（教科書の (2)）：
# run(f"unet_seg.py --seeds 0 1 2 --label {LABEL} --outdir {R} --price-usd-h {PRICE}")

In [ ]:
# 確かめ（教科書 III-6-5 の手順の「確かめること」。試しの設定でも本番でも成り立つことだけ）
j = load(f"{R}/unet_{LABEL}.json")
t = [r["train_s"] for r in j["runs"]]
m, lo, hi = j["train_s_mean_ci"]
dm, dlo, dhi = j["dice_mean_ci"]
ok(len(t) == len(j["args"]["seeds"]) >= 2 and lo <= m <= hi and 0 <= dlo <= dm <= dhi <= 1,
   f"[手順4] 学習時間 {m:.1f} 秒［{lo:.1f}, {hi:.1f}］、Dice {dm:.3f}［{dlo:.3f}, {dhi:.3f}］（種 {len(t)} 個）")
ok(all(len(r["curve"]) == j["args"]["epochs"] for r in j["runs"]), "どの種にも学習曲線（エポックごとの検証 Dice）がある")

**CPU だけの場合**（教科書の (3)。1 エポックで 1 ステップの時間を見る）。GPU と比べて何倍速いかを読みます。既定は課題1・2 とも小さいデータです（教科書の (3) は本番のデータで 1 エポック。Mac の CPU で課題1 が約 12 秒、課題2 が約 73 秒）。Colab の CPU は 2 コアほどのことが多い（公式で要確認）ので、Mac の CPU より遅いことがあります。

In [ ]:
run(f"anomaly_ae.py --seeds 0 --epochs 1 --n-train 1000 --n-test 400 --device cpu --label {LABEL_CPU} --outdir {R}")
run(f"unet_seg.py --seeds 0 --epochs 1 --n-train 320 --n-val 64 --device cpu --label {LABEL_CPU} --outdir {R}")
# 本番（教科書の (3)）：
# run(f"anomaly_ae.py --seeds 0 --epochs 1 --device cpu --label {LABEL_CPU} --outdir {R}")
# run(f"unet_seg.py --seeds 0 --epochs 1 --device cpu --label {LABEL_CPU} --outdir {R}")

In [ ]:
# 確かめ（教科書 III-6-5 の手順の「確かめること」。試しの設定でも本番でも成り立つことだけ）
ja, ju = load(f"{R}/ae_{LABEL_CPU}.json"), load(f"{R}/unet_{LABEL_CPU}.json")
ok(ja["env"]["device"] == "cpu" and ju["env"]["device"] == "cpu", "--device cpu で、CPU だけで計算した")
g, c = load(f"{R}/ae_{LABEL}.json"), ja
sg, sc = np.median([r["step_med_s"] for r in g["runs"]]), c["runs"][0]["step_med_s"]
print(f"課題1の 1 ステップ（中央値）：{DEV.type} {1e3 * sg:.1f} ms、CPU {1e3 * sc:.1f} ms → {sc / sg:.1f} 倍")
print(f"課題1の推論：{DEV.type} {np.mean([r['infer_per_s'] for r in g['runs']]):.0f} 個/秒、CPU {c['runs'][0]['infer_per_s']:.0f} 個/秒")
ok(sg > 0 and sc > 0, "GPU と CPU の 1 ステップの時間を比べられた（倍率は機械ごとに違う。III-6-7 の表と比べる）")

## 3. 課題3：NMF を PyTorch に移す（III-6-3-5、`nmf_torch.py`）

**速さを測る前に、答えが同じかを確かめます**（手順 3）。同じ初期値から 200 回更新し、NumPy（float64）版との相対差 $\|A-B\|_F/\|B\|_F$ を見ます。合格の目安は、float64 どうしで $10^{-10}$ 以下、float32（torch の CPU と GPU）で $WH$ が $10^{-3}$ 以下です（KL と IS の両方）。数秒で終わります。

In [ ]:
run(f"nmf_torch.py check --label {LABEL} --outdir {R}")

In [ ]:
# 確かめ（教科書 III-6-5 の手順の「確かめること」。試しの設定でも本番でも成り立つことだけ）
j = load(f"{R}/nmf_check_{LABEL}.json")
rows = {(x["div"], x["impl"]): x for x in j["rows"]}
for div in ("kl", "is"):
    f64 = rows[(div, "cpu float64")]
    ok(max(f64["rel_W"], f64["rel_H"], f64["rel_WH"]) < 1e-10, f"[手順3] {div.upper()}：float64 どうしの相対差が 1e-10 以下（W {f64['rel_W']:.1e}・H {f64['rel_H']:.1e}・WH {f64['rel_WH']:.1e}）")
    for impl in ("cpu float32", f"{DEV.type} float32"):
        x = rows[(div, impl)]
        ok(x["rel_WH"] < 1e-3, f"[手順3] {div.upper()}：{impl} の WH の相対差が 1e-3 以下（{x['rel_WH']:.1e}）")
ok(j["iters"] == 200 and len(j["rows"]) == 6, "KL・IS × 3 通り（cpu float64・cpu float32・GPU float32）を 200 反復で照合した")

速さを測ります（手順 4）。1 回の測定で 20 反復を、ウォームアップ 1 回のあと `--reps` 回測った中央値を 1 反復あたりに直します。NumPy（float64）、torch の CPU（float32）、GPU（float32）の 3 つを並べるので、「GPU が NumPy の○倍」のうち、float64 と float32 の違いの分も分けて読めます。

**既定は小さい設定**（大きさの表の最初の 3 段階＝ 1025×1000 まで、各 3 回）。教科書の本番は 5 段階（2049×8000 まで）を各 5 回です。

In [ ]:
NMF_REPS = 3
run(f"nmf_torch.py bench --label {LABEL} --outdir {R} --max-size 3 --reps {NMF_REPS}")
# 本番（教科書の (2)）：
# run(f"nmf_torch.py bench --label {LABEL} --outdir {R}")

In [ ]:
# 確かめ（教科書 III-6-5 の手順の「確かめること」。試しの設定でも本番でも成り立つことだけ）
j = load(f"{R}/nmf_bench_{LABEL}.json")
n_sizes = len(j["rows"])
ok(n_sizes >= 2, f"[手順4] NMF を {n_sizes} 段階の大きさで測った")
bad = []
for x in j["rows"]:
    med = {k: float(np.median(x[k])) for k in ("numpy_s_per_iter", "torchcpu_s_per_iter", "gpu_s_per_iter")}
    if not (all(v > 0 for v in med.values()) and abs(x["speedup_vs_numpy"] - med["numpy_s_per_iter"] / med["gpu_s_per_iter"]) < 1e-9):
        bad.append(x)
ok(not bad, f"どの大きさでも、倍率 ＝ NumPy の中央値 ÷ GPU の中央値（1 反復あたり）{[round(x['speedup_vs_numpy'], 2) for x in j['rows']]}")

## 4. 課題4：弦の振動の差分法（III-6-3-6、`string_fd.py`）

照合（手順 3）：(1) 格子を細かくしたときの誤差（$N$ を 2 倍で約 1/4 ＝ 2 次精度）、(2) クーラン数 $r$ を変える（$r = 1.02$ で発散＝ CFL 条件）、(3) NumPy（float64）と GPU（float32）の差（目安 $10^{-4}$ 以下）。数秒で終わります。

In [ ]:
run(f"string_fd.py check --label {LABEL} --outdir {R}")

In [ ]:
# 確かめ（教科書 III-6-5 の手順の「確かめること」。試しの設定でも本番でも成り立つことだけ）
j = load(f"{R}/string_check_{LABEL}.json")
conv, cfl_rows = j["rows"][:4], j["rows"][4:]          # string_fd.py check の (1) 4 行と (2) 5 行
ratio = [a["max_err"] / b["max_err"] for a, b in zip(conv, conv[1:])]
ok([x["N"] for x in conv] == [51, 101, 201, 401] and all(3.5 < q < 4.5 for q in ratio), f"[手順3] N を 2 倍にすると誤差が約 1/4（比 {[round(q, 2) for q in ratio]}。2次精度）")
cfl = {x["r"]: x["max_err"] for x in cfl_rows}
ok(all(cfl[r] < 1e-2 for r in (0.5, 0.9, 1.0)) and cfl[1.02] > 1 and cfl[1.1] > 1,
   f"[手順3] r ≤ 1 では誤差が小さく、r = 1.02・1.1 で発散（{cfl[1.02]:.1e}・{cfl[1.1]:.1e}。CFL 条件）")
ok(cfl[1.0] < cfl[0.5], f"r = 1 の誤差（{cfl[1.0]:.1e}）は r = 0.5 より小さい（格子の上で誤差 0 になる特別な場合。残りは 1e-9 の乱れと丸め）")
ok(j["np_vs_gpu_max"] < 1e-4, f"[手順3] NumPy（float64）と {DEV.type}（float32）の差が 1e-4 以下（{j['np_vs_gpu_max']:.1e}）")

速さを測ります（手順 4）。弦 $B$ 本 × 格子 $N$ 点をまとめて 200 歩進め、1 歩あたりの時間の中央値を比べます。演算強度が小さい（メモリの帯域で速さが決まる）計算なので、NMF とは倍率の上がり方が違うはずです（III-6-3-1）。

**既定は小さい設定**（最初の 4 段階＝ 256×4001 まで、各 3 回）。教科書の本番は 6 段階（1024×16001 まで）を各 5 回です。

In [ ]:
STR_REPS = 3
run(f"string_fd.py bench --label {LABEL} --outdir {R} --max-size 4 --reps {STR_REPS}")
# 本番（教科書の (2)）：STR_REPS = 5 にして
# run(f"string_fd.py bench --label {LABEL} --outdir {R}")

In [ ]:
# 確かめ（教科書 III-6-5 の手順の「確かめること」。試しの設定でも本番でも成り立つことだけ）
j = load(f"{R}/string_bench_{LABEL}.json")
ok(len(j["rows"]) >= 2 and all(len(x["gpu_s"]) == STR_REPS for x in j["rows"]), f"[手順4] 弦を {len(j['rows'])} 段階の大きさで、各 {STR_REPS} 回測った")
bad = [x for x in j["rows"] if not (np.median(x["numpy_s"]) > 0 and np.median(x["gpu_s"]) > 0
       and abs(x["speedup_vs_numpy"] - np.median(x["numpy_s"]) / np.median(x["gpu_s"])) < 1e-9)]
ok(not bad, f"どの大きさでも、倍率 ＝ NumPy の中央値 ÷ GPU の中央値（1 歩あたり）{[round(x['speedup_vs_numpy'], 2) for x in j['rows']]}")

## 5. 表にまとめ、損益分岐点と費用を読む（手順 4・5・8、`p3c6_table.py`）

`p3c6_table.py` で、このマシンの結果を 1 つの表にします（教科書の (4)）。Mac の結果（`runs/p3c6`）と並べるときは、下の最後のセルで結果を持ち帰り、Mac で `uv run --locked python p3c6_table.py runs/p3c6 runs/p3c6_colab_t4` のようにフォルダを並べます（machine の列でどのマシンか分かります）。

In [ ]:
out = run(f"p3c6_table.py {R}")

In [ ]:
# 確かめ（教科書 III-6-5 の手順の「確かめること」。試しの設定でも本番でも成り立つことだけ）
for h in ("■ 課題1 異常検知", "■ 課題2 U-Net", "■ 課題3 NMF 1反復", "■ 課題4 弦 1歩"):
    ok(h in out, f"[手順4] 表に「{h[2:]}」がある")
ok(out.count("GPU が NumPy より速くなった最小の大きさ") >= 2 and "_quick" not in out, "NMF と弦の損益分岐の行があり、--quick の結果は表に入らない")

**損益分岐点を読む**（手順 5）。III-6-3-1 の一次式 $t(n) = a + bn$ を、表の 2 点（いちばん小さい大きさと、いちばん大きい大きさ）から CPU（NumPy）と GPU のそれぞれに当てはめ、$n^{*} = (a_g - a_c)/(b_c - b_g)$ を計算して、表の「GPU が NumPy より速くなった最小の大きさ」と比べます（ノートブック版で足したセル。実際の曲線は直線ではないので、目安です）。図は、横軸が大きさ（百万要素・百万点）、縦軸が 1 反復・1 歩の時間です（両対数）。

In [ ]:
# 損益分岐点（ノートブック版で足したもの）
import matplotlib.pyplot as plt
import logging, warnings
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message="Glyph")
plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # 日本語のフォントは 0-1 のセルで用意する（見つからなければそこで知らせる）


def breakeven(a_c, b_c, a_g, b_g):
    """t_CPU = a_c + b_c n と t_GPU = a_g + b_g n が交わる n*（b_c <= b_g なら交わらない＝ GPU は大きくしても勝たない）"""
    return (a_g - a_c) / (b_c - b_g) if b_c > b_g else float("inf")


def fit2(n, t):
    """2 点 (n[0], t[0])・(n[-1], t[-1]) を通る直線の a, b"""
    b = (t[-1] - t[0]) / (n[-1] - n[0])
    return t[0] - b * n[0], b


fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, fn, keys, unit, scale) in zip(axes, [
        ("課題3 NMF（1 反復、ms）", "nmf_bench", ("numpy_s_per_iter", "torchcpu_s_per_iter", "gpu_s_per_iter"), "百万要素", 1e3),
        ("課題4 弦（1 歩、µs）", "string_bench", ("numpy_s", "torchcpu_s", "gpu_s"), "百万点", 1e6)]):
    j = load(f"{R}/{fn}_{LABEL}.json")
    rows = j["rows"]
    n = np.array([(x["F"] * x["T"]) if "F" in x else (x["B"] * x["N"]) for x in rows]) / 1e6
    per = [1 if "iters" in x else x["steps"] for x in rows]
    med = {k: np.array([scale * np.median(x[k]) / p for x, p in zip(rows, per)]) for k in keys}
    for k, lab in zip(keys, ["NumPy", "torch CPU f32", f"{DEV.type} f32"]):
        ax.loglog(n, med[k], "o-", label=lab)
    ac, bc = fit2(n, med[keys[0]])
    ag, bg = fit2(n, med[keys[2]])
    ns = breakeven(ac, bc, ag, bg)
    first = next((f"{m:.2f}" for m, x in zip(n, rows) if x["speedup_vs_numpy"] > 1), "この範囲には無い")
    print(f"{name}：2 点からの一次式で n* = {ns:.3g} {unit}（a_c {ac:.3g}、b_c {bc:.3g}、a_g {ag:.3g}、b_g {bg:.3g}）、"
          f"実測で GPU が NumPy に勝った最小の大きさ {first} {unit}")
    ax.set_title(name)
    ax.set_xlabel(f"大きさ（{unit}）")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# 確かめ（教科書 III-6-5 の手順の「確かめること」。試しの設定でも本番でも成り立つことだけ）
ok(abs(breakeven(0.0, 1.0, 0.25, 0.1) - 0.25 / 0.9) < 1e-12, "III-6-3-1 の例：n* = 0.25/0.9 ≈ 0.28 百万要素")
ok(abs(breakeven(0.05, 2.0, 0.40, 0.25) - 0.2) < 1e-12, "確認問題1：n* = 0.35/1.75 = 0.2 百万要素")

**費用を計算する**（手順 8、III-6-3-2 の式）。学習の課題（課題1・2）の学習時間の平均に、準備と片付けの時間を足して計算します。下のセルは、このノートブックの準備のセルからいままでの経過時間を「準備＋待ち時間を含めた全体」として表示します。Colab では、ランタイムの接続からこのノートブックを開くまでの時間と、終わったあとに接続を切るまでの時間も費用に入ります（`PRICE` が 0 の無料の枠でも、使える時間の残りが減ります）。

小さな課題をクラウドで回すと、費用のほとんどは待ち時間です。4 題をまとめて 1 回で行うのはこのためです。

In [ ]:
def cost_yen(t_prep_s, t_train_s, t_clean_s, price_usd_h, usd_jpy):
    """III-6-3-2：費用[円] = (t_準備 + m t̄ + t_片付け)/3600 × p × r"""
    return (t_prep_s + t_train_s + t_clean_s) / 3600 * price_usd_h * usd_jpy


elapsed = time.time() - T_NB_START
for task, fn in (("課題1", "ae"), ("課題2", "unet")):
    j = load(f"{R}/{fn}_{LABEL}.json")
    t = [r["train_s"] for r in j["runs"]]
    m, lo, hi = p3c6_util.mean_ci(t)
    print(f"{task}：学習時間 平均 {m:.1f} 秒［{lo:.1f}, {hi:.1f}］× 種 {len(t)} 個 → 学習だけの費用 {cost_yen(0, sum(t), 0, PRICE, USD_JPY):.2f} 円")
print(f"このノートブックの準備から今までの経過時間 {elapsed / 60:.1f} 分 → {cost_yen(elapsed, 0, 0, PRICE, USD_JPY):.2f} 円"
      f"（{PRICE} USD/時、{USD_JPY} 円/USD。片付けの時間は含まない）")
print("例：準備 20 分＋片付け 5 分を 0.40 USD/時で計算すると", cost_yen(20 * 60, 0, 5 * 60, 0.40, 150), "円")

In [ ]:
# 確かめ（教科書 III-6-5 の手順の「確かめること」。試しの設定でも本番でも成り立つことだけ）
ok(abs(cost_yen(20 * 60, 0, 5 * 60, 0.40, 150) - 25.0) < 1e-9, "III-6-3-2 の例：準備 20 分＋片付け 5 分、0.40 USD/時で 25 円")
ok(abs(cost_yen(20 * 60, 3 * 10, 5 * 60, 0.40, 150) - 25.5) < 1e-9, "確認問題6：学習 10 秒 × 3 回を足して 25.5 円")
m3 = p3c6_util.mean_ci([20.3, 22.1, 19.2])
ok(abs(m3[0] - 20.53) < 0.01 and abs(m3[1] - 16.9) < 0.05 and abs(m3[2] - 24.2) < 0.05, f"確認問題2：平均 20.53 秒、95% 区間［16.9, 24.2］（{m3[1]:.2f}, {m3[2]:.2f}）")

**結果を持ち帰る**（教科書の手順 7 の `vast_sync.sh pull` の代わり）。Colab のファイルはランタイムを切ると消えるので、`R` のフォルダを zip にしてダウンロードします（Mac で動かしているときは何もしません）。Mac では `~/home-cloud/runs/` の下に展開し、`p3c6_table.py runs/p3c6 runs/p3c6_colab_t4` のように Mac の結果と並べます。Google Drive に残したいときは、III-2 のセル2 のように Drive をマウントしてコピーしてください。

終わったら、Colab のランタイムを解放します（「ランタイム」メニューの「ランタイムを接続解除して削除」など。項目の名前は公式ドキュメントで要確認。III-2 のセル7 の `runtime.unassign()` でもかまいません）。

In [ ]:
import shutil
if IN_COLAB:
    zip_path = shutil.make_archive(f"p3c6_{MACHINE}", "zip", root_dir="runs", base_dir=os.path.basename(R))
    from google.colab import files
    files.download(zip_path)
    # from google.colab import runtime; runtime.unassign()   # 結果をダウンロードしてから。ランタイムを返す（III-2 のセル7）
else:
    print(f"Colab ではないので、持ち帰りはしません。結果は {os.path.abspath(R)} にあります")
print(sorted(os.listdir(R)))

## おわりに

- 4 題（異常検知・U-Net・NMF・弦の差分法）を、「小さく動作確認 → 答えの照合 → 本番と同じ設定で測る → 表にまとめる」の順に通しました。学習の課題では「精度が同じで、時間と費用がどれだけ違うか」を、計算の課題では「どの大きさから GPU が得になるか（損益分岐点）」を数字で読みます。
- 本番の設定で測るときは、各セルのコメントの行に替えて 1 節から実行し直してください（Colab の T4 などで、全部で十数分〜数十分の見込み。データの合成が CPU なので、GPU が速くても学習時間はあまり縮まないことがあります。そのときは教科書のトラブルの表の「GPU-Util が低い」を見てください）。
- 手順 8 の「4 題それぞれについて、Mac で十分か、クラウドが得か、どの規模からか」を 1〜2 文でノートに書き、Mac とクラウドで精度（AUC・Dice）が区間の中で同じであることも確かめてください。